# 01 — Document ingestion and parsing: Steps 1–8

This notebook implements only **source loading, DOCX inventory, parser setup, ordered structural block extraction, conservative document/region/marker detection, assembly of the intermediate corpus representation, metadata provenance classification, and validated Stage 1 persistence** from [the approved plan](../docs/stage-01-ingestion-parsing-plan.md).

Run from top to bottom using the project's `.venv` kernel. Setup from the project directory: `python -m pip install -e '.[dev]'`.

We read the combined DOCX directly from its archive, inspect the stored structure, compare a **small sample** with `python-docx`, then extract ordered structural records in Step 4 add separate evidence-backed annotations in Step 5, assemble them into one corpus record in Step 6, distinguish metadata origins in Step 7, and validate/persist the result in Step 8. Steps 1–7 stay in memory; Step 8 writes only the processed JSON artifact after validation passes. Reference chunks and evaluation data are not read.

Each section explains what, why, how, and what to inspect. Preview truncation affects display only. This is structural inspection, not cleaning or a complete parsing pipeline.

## Step 1 — Locate and load the exact source

**What/why:** Select the exact combined DOCX so reference chunks and companion files cannot accidentally become input.

**How:** Resolve the project root from either the project or notebook directory. Read only the named ZIP member into `BytesIO`; do not extract the archive. Hash the same bytes that the parser will see.

**Choice:** In-memory loading avoids an extra copy on disk. Extraction would help manual Word viewing but is unnecessary here. A hash identifies this file snapshot, not a legal instrument. Missing, duplicate, or corrupt source entries should fail visibly.

In [ ]:
from pathlib import Path
from io import BytesIO
from hashlib import sha256
from zipfile import ZipFile, BadZipFile
from collections import Counter
from itertools import islice
from importlib.metadata import version
import xml.etree.ElementTree as ET
from html import escape
from IPython.display import display, HTML

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").is_file()
             and (p / "src/property_copilot").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter in the project or notebooks directory")
ARCHIVE = ROOT / "data/raw/bg_legal_corpus_real.zip"
MEMBER = "bg_legal_corpus_real/full_text/ALL_FULL_TEXTS.docx"

def load_source(archive, member):
    if not archive.is_file():
        raise FileNotFoundError(f"Source archive is missing: {archive}")
    try:
        with ZipFile(archive) as bundle:
            matches = [entry for entry in bundle.infolist() if entry.filename == member]
            if len(matches) != 1:
                raise ValueError(f"Expected exactly one {member}; found {len(matches)}")
            return bundle.read(matches[0])
    except BadZipFile as exc:
        raise ValueError("Source archive is not a valid ZIP") from exc

docx_bytes = load_source(ARCHIVE, MEMBER)
manifest = {"archive": ARCHIVE.relative_to(ROOT).as_posix(),
            "member": MEMBER, "docx_bytes": len(docx_bytes),
            "sha256": sha256(docx_bytes).hexdigest()}
display(manifest)


**Inspect:** The member must end in `full_text/ALL_FULL_TEXTS.docx`; the inspected snapshot is 940,203 bytes. No timestamp is manufactured. The SHA-256 enables comparison on later reruns.

## Step 2 — Inventory the DOCX package

**What/why:** A DOCX is itself a ZIP containing XML parts. Inspect these before relying on a higher-level parser, which may not expose every Word feature.

**How:** Read document XML and style definitions with the standard library. Inventory all package names but do not load unrelated archive members. XML namespaces distinguish Word elements from other formats.

**Choice:** XML is explicit but verbose. We use it for counts and representative inspection, not to build a custom full parser.

In [ ]:
NS = {"w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main"}
W = "{" + NS["w"] + "}"
try:
    with ZipFile(BytesIO(docx_bytes)) as package:
        part_names = package.namelist()
        required = ["[Content_Types].xml", "word/document.xml", "word/styles.xml"]
        for name in required:
            if part_names.count(name) != 1:
                raise ValueError(f"DOCX requires exactly one {name}")
        document_xml = ET.fromstring(package.read("word/document.xml"))
        styles_xml = ET.fromstring(package.read("word/styles.xml"))
        auxiliary_xml = {name: ET.fromstring(package.read(name)) for name in part_names
                         if name.startswith(("word/header", "word/footer"))
                         and name.endswith(".xml")}
except (BadZipFile, ET.ParseError) as exc:
    raise ValueError("Selected member is not a readable DOCX XML package") from exc
body = document_xml.find("w:body", NS)
if body is None:
    raise ValueError("DOCX has no Word document body")
body_elements = list(body)  # Raw XML elements for inspection, not parsed block records.
print("DOCX package parts:")
print("\n".join(part_names))


In [ ]:
def show_rows(rows):
    """Small escaped HTML table; no dataframe dependency or source HTML execution."""
    if not rows:
        print("No rows")
        return
    columns = list(rows[0])
    header = "".join(f"<th>{escape(str(k))}</th>" for k in columns)
    content = "".join("<tr>" + "".join(
        f"<td style='white-space:pre-wrap'>{escape(str(row.get(k, '')))}</td>"
        for k in columns) + "</tr>" for row in rows)
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{content}</tbody></table>"))

def text_preview(element):
    # Inspection only: w:t concatenation does not represent tabs, breaks, or fields.
    return "".join(node.text or "" for node in element.iter(W + "t"))

def style_id(paragraph):
    node = paragraph.find("w:pPr/w:pStyle", NS)
    return node.get(W + "val") if node is not None else "(default)"

paragraphs_xml = body.findall("w:p", NS)
tables_xml = body.findall("w:tbl", NS)
style_names = {s.get(W + "styleId"): s.find("w:name", NS).get(W + "val")
               for s in styles_xml.findall("w:style", NS)
               if s.find("w:name", NS) is not None}
default_paragraph_style = next(
    s.get(W + "styleId") for s in styles_xml.findall("w:style", NS)
    if s.get(W + "type") == "paragraph" and s.get(W + "default") == "1")
style_counts = Counter(style_id(p) for p in paragraphs_xml)
counts = {
    "top-level paragraphs": len(paragraphs_xml),
    "top-level tables": len(tables_xml),
    "all body/table paragraphs": len(list(body.iter(W + "p"))),
    "table rows": len(list(body.iter(W + "tr"))),
    "table cells": len(list(body.iter(W + "tc"))),
    "Heading1": style_counts["Heading1"],
    "Heading2": style_counts["Heading2"],
    "Heading3": style_counts["Heading3"],
    "explicit page breaks": sum(b.get(W + "type") == "page" for b in body.iter(W + "br")),
    "section properties": len(list(body.iter(W + "sectPr"))),
}
expected = dict(zip(counts, [31055, 21, 31307, 126, 252, 22, 509, 195, 41, 1]))
show_rows([{"measure": k, "observed": v, "inspected snapshot": expected[k],
            "matches": v == expected[k]} for k, v in counts.items()])
print("Top-level XML types:", dict(Counter(e.tag.rsplit('}', 1)[-1] for e in body)))
show_rows([{"applied style ID": s, "style name": style_names.get(s, "default paragraph style"),
            "paragraph count": count} for s, count in style_counts.items()])
if counts != expected:
    print("REVIEW: source structure differs from the approved plan's snapshot.")


**Interpretation:** 22 Heading 1 paragraphs include the corpus title and 21 source titles. One Word section is a layout setting, not 21 legal-document boundaries. The 252 cell paragraphs account for the difference between 31,055 top-level and 31,307 total paragraphs.

A style being defined in `styles.xml` does not mean it is used. Parts and chapters may both have Heading 2, so style depth alone cannot establish legal hierarchy. The 41 explicit page breaks do not tell us the rendered page count.

In [ ]:
# Small opening window around the first source title and metadata table.
first_table_index = next(i for i, e in enumerate(body_elements) if e.tag == W + "tbl")
show_rows([{"body index (zero-based)": i, "XML type": body_elements[i].tag.rsplit("}", 1)[-1],
            "style": style_id(body_elements[i]) if body_elements[i].tag == W + "p" else "—",
            "preview (220 chars)": text_preview(body_elements[i])[:220]}
           for i in range(first_table_index - 2, first_table_index + 2)])
first_table_xml = tables_xml[0]
show_rows([{"row": r, "label": text_preview(row.findall("w:tc", NS)[0]),
            "value": text_preview(row.findall("w:tc", NS)[1])}
           for r, row in enumerate(first_table_xml.findall("w:tr", NS))])
print("Table shapes (rows, cells per row):")
print(Counter((len(t.findall("w:tr", NS)),
               tuple(len(r.findall("w:tc", NS)) for r in t.findall("w:tr", NS)))
              for t in tables_xml))

# Locate examples only; do not assign regions, article spans, or document membership.
example_prefixes = ["Извлечени правила (reference)", "Пълен текст / Full text",
                    "Част първа", "Глава първа", "Раздел I", "Чл. 1.", "Чл. 2а.",
                    "Член 1", "Приложение", "§"]
examples = []
for prefix in example_prefixes:
    found = next(((i, e) for i, e in enumerate(body_elements)
                  if e.tag == W + "p" and text_preview(e).startswith(prefix)), None)
    if found:
        i, e = found
        examples.append({"example prefix": prefix, "body index": i,
                         "style": style_id(e), "preview (350 chars)": text_preview(e)[:350]})
show_rows(examples)
print("Representative heading XML:")
print(ET.tostring(body_elements[first_table_index - 2], encoding="unicode"))


**Inspect:** The metadata tables contain jurisdiction, URL, official reference, version date, access date, and notes. These are source claims, not verified legal facts. The full-text and reference-rule markers are distinct. Bulgarian `Чл.` provisions can be ordinary paragraphs while EU `Член` provisions can be headings.

Reference rules remain in the source untouched; their later labeling is Step 5. Here we only view examples. No metadata mapping or region assignment is implemented.

In [ ]:
# Features that a simple paragraph/table traversal could overlook.
review_tags = ["ins", "del", "sdt", "txbxContent", "drawing", "pict", "altChunk",
               "hyperlink", "footnoteReference", "endnoteReference", "commentReference",
               "fldSimple", "fldChar", "instrText", "numPr", "gridSpan", "vMerge"]
show_rows([{"feature": tag, "body occurrences": len(list(body.iter(W + tag)))}
           for tag in review_tags])
extra_parts = [p for p in part_names if p.startswith(
    ("word/footnotes", "word/endnotes", "word/comments", "word/media/", "word/embeddings/"))]
print("Additional content parts requiring review:", extra_parts or "none")
unexpected_body = sorted({e.tag for e in body if e.tag not in {W+'p', W+'tbl', W+'sectPr'}})
print("Unexpected top-level body elements:", unexpected_body or "none")
print("Nested tables:", len(list(body.iter(W + "tbl"))) - len(tables_xml))
for name, xml in auxiliary_xml.items():
    print("Auxiliary part:", name)
    print("Visible text:", repr(text_preview(xml)))
    print("Field instructions:", [e.text for e in xml.iter(W + "instrText")])
    print("Field events:", [e.get(W + "fldCharType") for e in xml.iter(W + "fldChar")])
print("Review note: this feature scan is not proof of complete Word rendering fidelity.")


**Interpretation:** The footer's `PAGE` instruction is a dynamic Word field; it has no usable rendered page value here. We will not invent page numbers. The scan highlights common omission risks and unknown body elements; it is not a universal DOCX validator.

## Step 3 — Set up the parser and compare a small region

**What/why:** Use `python-docx` for readable paragraph, style, run, and table objects, with XML inspection as an independent check. Its public `iter_inner_content()` preserves paragraph/table order. Some features, including paragraphs inside revision marks, need additional handling; see the [official API](https://python-docx.readthedocs.io/en/latest/api/document.html).

**How:** Open the same bytes and inspect only the first document's title, subtitle, table, and next heading. The dependency is `python-docx>=1.2,<2`; versions are printed below. The range allows compatible updates, while the recorded versions identify this run.

**Alternatives/trade-offs:** XML-only parsing offers more control but requires more Word-specific code. [docx2python](https://github.com/ShayHill/docx2python) supports additional content types but adds a nested representation. [Mammoth](https://github.com/mwilliamson/python-mammoth) provides readable HTML while intentionally discarding some formatting detail. Plain text loses the structure we need. We introduce no RAG framework.

**Inspect:** Compare text and style IDs (not localized display names), table dimensions, and every cell of this one metadata table. This is a bounded demonstration, not Step 4's corpus-wide extraction.

An absent XML paragraph style means “use the default,” not “no style.” We read the default paragraph style from `styles.xml` and compare it with the resolved style returned by `python-docx`. The inventory still distinguishes explicit style assignments from defaults.

In [ ]:
from docx import Document
from docx.text.paragraph import Paragraph
from docx.table import Table
import sys

print("Python:", sys.version.split()[0])
print("python-docx:", version("python-docx"))
print("lxml:", version("lxml"))
parser_document = Document(BytesIO(docx_bytes))

# islice limits the demonstration; no full collection of parsed blocks is built.
sample_start = first_table_index - 2
sample_end = first_table_index + 2
sample = list(islice(parser_document.iter_inner_content(), sample_start, sample_end))
xml_sample = body_elements[sample_start:sample_end]
checks = []
for i, (item, xml) in enumerate(zip(sample, xml_sample), start=sample_start):
    if isinstance(item, Paragraph):
        text_matches = item.text == text_preview(xml)
        explicit_style = style_id(xml)
        expected_style = default_paragraph_style if explicit_style == "(default)" else explicit_style
        style_matches = item.style.style_id == expected_style
        checks.append({"body index": i, "kind": "paragraph", "text matches": text_matches,
                       "style/dimensions match": style_matches})
        assert text_matches and style_matches, f"Paragraph mismatch at {i}"
        print(f"{i}: {item.style.name} / {item.style.style_id}: {item.text}")
    elif isinstance(item, Table):
        xml_rows = xml.findall("w:tr", NS)
        api_cells = [[cell.text for cell in row.cells] for row in item.rows]
        xml_cells = [["\n".join(text_preview(p) for p in cell.findall("w:p", NS))
                      for cell in row.findall("w:tc", NS)] for row in xml_rows]
        dimensions_match = len(item.rows) == len(xml_rows) and all(
            len(a) == len(b) for a, b in zip(api_cells, xml_cells))
        checks.append({"body index": i, "kind": "table", "text matches": api_cells == xml_cells,
                       "style/dimensions match": dimensions_match})
        assert api_cells == xml_cells and dimensions_match, "Metadata table mismatch"
        show_rows([{"row": r, "column": c, "python-docx cell text": value}
                   for r, row in enumerate(api_cells) for c, value in enumerate(row)])
    else:
        raise AssertionError(f"Unexpected sample type: {type(item)}")
assert len(sample) == len(xml_sample) == 4
show_rows(checks)
print("Small-region comparison passed: 3 paragraphs and one 6 × 2 metadata table.")


## Step 4 — Extract ordered structural blocks

**What:** Create `ordered_blocks`: one record for each body paragraph or table, in source order. Tables contain ordered rows, cells, and paragraph records. `auxiliary_content` holds footer/header records separately; `layout_markers` accounts for Word section properties.

**Why:** Separate paragraph and table lists lose their interleaving. A single text string loses formatting, empty paragraphs, and table coordinates. We preserve these structures so you can inspect them before deciding any later processing.

**How:** Pair `python-docx.iter_inner_content()` with the independent XML inventory. The public API supplies paragraph/run text and formatting; XML supplies exact structural positions and explicit break/field events. Every body element must be accounted for, or extraction stops with a location-specific error.

**Scope:** These are Step 4 block records, not Step 6's complete corpus/document representation. No document IDs, regions, legal roles, inferred metadata, or chunk relationships are assigned. Nothing is exported.

### 4a. Record conventions and source locations

| Field | Meaning |
|---|---|
| `id` | SHA-256 of source-file hash, XML part, and structural path |
| `position` | Zero-based child position within its immediate container |
| `source` | XML part and zero-based structural path; this is a locator convention, not XPath |
| `raw_text` | Paragraph text returned by the parser, without trimming or normalization |
| `style` | Explicit style ID and resolved default ID/name; no semantic interpretation |
| `runs` | Ordered run text, direct formatting, and ordered XML content events |
| `break_markers` | Break type and its run/event position, including invisible page breaks |
| `rows` | For tables: ordered cells containing their paragraph records |

A cell paragraph location looks like `word/document.xml` + `/body/children/26/rows/0/cells/1/children/1`. IDs are reproducible for identical source bytes and locations, not permanent legal identifiers across revisions.

**Text choice:** Text runs keep spaces and Unicode exactly. `raw_text` follows Word parser semantics: a tab is `\t`, a line break is `\n`, and a page/column break or field instruction has no visible character. The ordered events preserve those distinctions; no paragraph separators are inserted. A table has no canonical flattened text.

**Formatting choice:** `None` in bold/italic/etc. means “not directly specified,” not false. We retain direct formatting and serialized property XML instead of computing Word's full inherited appearance. Serialized XML preserves element information, not byte-for-byte XML formatting.

In [ ]:
def base_record(kind, position, part, path, source_hash):
    identity = f"{source_hash}:{part}:{path}"
    return {"id": sha256(identity.encode("utf-8")).hexdigest(),
            "type": kind, "position": position,
            "source": {"part": part, "path": path}}

# Cache definitions once: resolving styles repeatedly across 31,000 paragraphs is unnecessary.
style_catalog = {s.style_id: s.name for s in parser_document.styles}
default_styles = {
    s.get(W + "type"): s.get(W + "styleId")
    for s in styles_xml.findall("w:style", NS) if s.get(W + "default") == "1"
}

def source_style(xml, kind):
    style_path = "w:pPr/w:pStyle" if kind == "paragraph" else "w:tblPr/w:tblStyle"
    node = xml.find(style_path, NS)
    explicit = node.get(W + "val") if node is not None else None
    resolved = explicit if explicit is not None else default_styles.get(kind)
    return {"explicit_id": explicit, "resolved_id": resolved,
            "name": style_catalog.get(resolved)}

def property_xml(element):
    return ET.tostring(element, encoding="unicode") if element is not None else None

def run_events(run_xml):
    events = []
    for child in run_xml:
        if child.tag == W + "rPr":
            continue
        if child.tag == W + "t":
            events.append({"type": "text", "text": child.text or ""})
        elif child.tag == W + "tab":
            events.append({"type": "tab"})
        elif child.tag in {W + "br", W + "cr"}:
            events.append({"type": "break", "xml_tag": child.tag.rsplit("}", 1)[-1],
                           "kind": child.get(W + "type", "textWrapping"),
                           "clear": child.get(W + "clear")})
        elif child.tag == W + "fldChar":
            events.append({"type": "field_char", "kind": child.get(W + "fldCharType"),
                           "xml": property_xml(child)})
        elif child.tag == W + "instrText":
            events.append({"type": "field_instruction", "text": child.text or ""})
        else:
            # Preserve unfamiliar run content visibly rather than silently dropping it.
            events.append({"type": "unhandled_xml", "tag": child.tag,
                           "xml": property_xml(child)})
    return events


### 4b. Paragraphs, runs, and explicit breaks

**What/how:** Keep each paragraph even when its visible text is empty. Pair runs with their XML siblings and preserve event order within each run. Record bold, italic, underline, strike, superscript/subscript, font name, and size; full direct run properties retain additional details such as color.

**Trade-off:** This extractor targets the inspected source's direct paragraph/run structure. It fails on unexpected paragraph containers instead of silently omitting tracked changes or content controls. Unhandled run elements remain visible as XML events and are counted in the checks below. This is not a universal Word renderer.

**Inspect next:** A page-break-only paragraph should have empty `raw_text` but a nonempty `break_markers` list. No filtering of empty strings is permitted.

In [ ]:
def paragraph_record(paragraph, xml, position, part, path, source_hash):
    unexpected = [e.tag for e in xml if e.tag not in {W + "pPr", W + "r"}]
    if unexpected:
        raise ValueError(f"Review unsupported paragraph content at {part}:{path}: {unexpected}")
    record = base_record("paragraph", position, part, path, source_hash)
    record.update(raw_text=paragraph.text, style=source_style(xml, "paragraph"),
                  paragraph_properties_xml=property_xml(xml.find("w:pPr", NS)),
                  runs=[], break_markers=[])
    xml_runs = xml.findall("w:r", NS)
    for run_index, (run, run_xml) in enumerate(zip(paragraph.runs, xml_runs, strict=True)):
        underline = run.underline
        if underline is not None and not isinstance(underline, bool):
            underline = underline.name
        formatting = {"bold": run.bold, "italic": run.italic, "underline": underline,
                      "strike": run.font.strike, "superscript": run.font.superscript,
                      "subscript": run.font.subscript, "font_name": run.font.name,
                      "font_size_pt": run.font.size.pt if run.font.size is not None else None}
        events = run_events(run_xml)
        record["runs"].append({"position": run_index, "raw_text": run.text,
                               "formatting": formatting,
                               "properties_xml": property_xml(run_xml.find("w:rPr", NS)),
                               "events": events})
        for event_index, event in enumerate(events):
            if event["type"] == "break":
                record["break_markers"].append({"run_index": run_index,
                    "event_index": event_index, "kind": event["kind"],
                    "xml_tag": event["xml_tag"], "clear": event["clear"]})
    return record


### 4c. Tables remain in place, with nested cell paragraphs

**What/how:** Store each table at its original body position. Iterate rows and cells once; each cell holds its own ordered paragraph records using the same paragraph function.

**Choice:** Preserve the six-by-two tables as cells, not dictionaries of inferred metadata. Coordinates express structure only. The observed tables are rectangular, unmerged, and unnested. Explicit checks reject merged, nested, or omitted-cell layouts requiring another representation, rather than duplicating or misplacing their content. General support for those layouts is unnecessary for this source.

**Inspect next:** A title paragraph, subtitle paragraph, table, and following paragraph must appear in that order. The first table's cell text is shown with row and column coordinates.

In [ ]:
def table_record(table, xml, position, part, path, source_hash):
    if list(xml.iter(W + "gridSpan")) or list(xml.iter(W + "vMerge")):
        raise ValueError(f"Merged cells require review at {part}:{path}")
    if len(list(xml.iter(W + "tbl"))) != 1:
        raise ValueError(f"Nested tables require review at {part}:{path}")
    if any(e.tag not in {W + "tblPr", W + "tblGrid", W + "tr"} for e in xml):
        raise ValueError(f"Unexpected table content at {part}:{path}")
    record = base_record("table", position, part, path, source_hash)
    record.update(style=source_style(xml, "table"), rows=[])
    for row_index, (row, row_xml) in enumerate(zip(table.rows, xml.findall("w:tr", NS), strict=True)):
        if any(e.tag not in {W + "trPr", W + "tc"} for e in row_xml):
            raise ValueError(f"Unexpected row content at {part}:{path}/rows/{row_index}")
        if row.grid_cols_before or row.grid_cols_after:
            raise ValueError(f"Omitted grid cells require review at {part}:{path}/rows/{row_index}")
        row_record = {"row_index": row_index, "cells": []}
        for column_index, (cell, cell_xml) in enumerate(zip(row.cells, row_xml.findall("w:tc", NS), strict=True)):
            cell_path = f"{path}/rows/{row_index}/cells/{column_index}"
            if any(e.tag not in {W + "tcPr", W + "p"} for e in cell_xml):
                raise ValueError(f"Unexpected cell content at {part}:{cell_path}")
            cell_paragraphs = []
            api_paragraphs = iter(cell.paragraphs)
            for child_index, child in enumerate(cell_xml):
                if child.tag == W + "p":
                    cell_paragraphs.append(paragraph_record(next(api_paragraphs), child, child_index,
                        part, f"{cell_path}/children/{child_index}", source_hash))
            if next(api_paragraphs, None) is not None:
                raise AssertionError(f"Unaccounted cell paragraph at {cell_path}")
            row_record["cells"].append({"column_index": column_index, "paragraphs": cell_paragraphs})
        record["rows"].append(row_record)
    return record


def extract_container(items, xml_container, part, path, source_hash):
    """Keep source order; XML positions include non-content layout elements."""
    items = iter(items)
    blocks, layout = [], []
    for position, xml in enumerate(xml_container):
        location = f"{path}/children/{position}"
        if xml.tag == W + "sectPr":
            marker = base_record("section_properties", position, part, location, source_hash)
            marker["xml"] = property_xml(xml)
            layout.append(marker)
            continue
        if xml.tag not in {W + "p", W + "tbl"}:
            raise ValueError(f"Unsupported container element at {part}:{location}: {xml.tag}")
        item = next(items, None)
        if xml.tag == W + "p" and isinstance(item, Paragraph):
            blocks.append(paragraph_record(item, xml, position, part, location, source_hash))
        elif xml.tag == W + "tbl" and isinstance(item, Table):
            blocks.append(table_record(item, xml, position, part, location, source_hash))
        else:
            raise AssertionError(f"Parser/XML order mismatch at {part}:{location}")
    if next(items, None) is not None:
        raise AssertionError(f"Unaccounted parser content in {part}")
    return blocks, layout

ordered_blocks, layout_markers = extract_container(
    parser_document.iter_inner_content(), body, "word/document.xml", "/body", manifest["sha256"])
print("Body records:", dict(Counter(b["type"] for b in ordered_blocks)))
print("Separate layout markers:", len(layout_markers))


### 4d. Footer content stays outside the body

**What/why:** Preserve existing footer/header parts once each. Repeating a footer inside body text would change reading order and duplicate content.

**How:** Read existing package parts and wrap their existing paragraph/table elements with `python-docx`. This small lower-level adapter does not create headers or footers. The same record functions preserve the footer's empty visible text and ordered `PAGE` field events.

**Trade-off:** Word evaluates fields when rendering pages. We keep the field instruction and do not calculate a page number. Section-property XML remains in `layout_markers`; no legal document boundaries are derived from it.

In [ ]:
# Read existing parts only; avoid accessors that could create missing header/footer definitions.
parts_by_name = {str(part.partname).lstrip("/"): part for part in parser_document.part.package.parts}
auxiliary_content = []
for part_name, xml in auxiliary_xml.items():
    part = parts_by_name[part_name]
    # FooterPart/HeaderPart expose existing XML; wrapper constructors are isolated here.
    items = (Paragraph(child, part) if child.tag == W + "p" else Table(child, part)
             for child in part.element if child.tag in {W + "p", W + "tbl"})
    records, layout = extract_container(items, xml, part_name, "/" + xml.tag.rsplit("}", 1)[-1],
                                        manifest["sha256"])
    auxiliary_content.append({"part": part_name, "blocks": records, "layout_markers": layout})
show_rows([{"part": a["part"], "paragraph/table records": len(a["blocks"])}
           for a in auxiliary_content])


### 4e. Inspect the records

The summary below uses short previews only; the underlying `raw_text` is complete. Table character counts sum cell paragraphs without adding artificial separators. The expandable JSON viewer shows a selected record in full, including source location, run formatting, and events. Change `SELECTED_BODY_POSITION` and rerun the viewer cell to inspect another record.

Look for the table at body position 26, the retained page-break-only paragraph, and the footer's field instruction. Table-cell paragraph positions count original XML children, including the preceding `tcPr` property element; they are source coordinates, not a count of just paragraphs.

In [ ]:
import json

def table_paragraphs(block):
    return [p for row in block["rows"] for cell in row["cells"] for p in cell["paragraphs"]]

def preview_record(block):
    paragraphs = [block] if block["type"] == "paragraph" else table_paragraphs(block)
    preview = " | ".join(p["raw_text"] for p in paragraphs)[:160]
    return {"position": block["position"], "type": block["type"],
            "style": block["style"]["name"], "preview (display only)": preview,
            "characters": sum(len(p["raw_text"]) for p in paragraphs)}

show_rows([preview_record(b) for b in ordered_blocks
           if first_table_index - 3 <= b["position"] <= first_table_index + 3])
first_table_record = next(b for b in ordered_blocks if b["type"] == "table")
show_rows([{"row": row["row_index"], "column": cell["column_index"],
            "paragraph position": p["position"], "raw text": p["raw_text"]}
           for row in first_table_record["rows"] for cell in row["cells"] for p in cell["paragraphs"]])

def show_full_record(record, title):
    content = escape(json.dumps(record, ensure_ascii=False, indent=2))
    display(HTML(f"<details><summary>{escape(title)}</summary><pre>{content}</pre></details>"))

SELECTED_BODY_POSITION = first_table_index - 2
selected = next(b for b in ordered_blocks if b["position"] == SELECTED_BODY_POSITION)
show_full_record(selected, "Full selected paragraph record")
show_full_record(first_table_record, "Full first table record (nested cells and paragraphs)")
page_break_paragraph = next(b for b in ordered_blocks if b["type"] == "paragraph"
                            and b["raw_text"] == "" and b["break_markers"])
show_full_record(page_break_paragraph, "Empty visible text, preserved page-break event")
for auxiliary in auxiliary_content:
    show_full_record(auxiliary, "Separate auxiliary content: " + auxiliary["part"])


### 4f. Check Step 4 preservation

**What/why:** Counts alone can pass even if text or ordering changes. These focused checks compare every body/cell paragraph with independent XML, verify identifiers and source positions, and reconcile all explicit breaks. They validate this extraction step; they do not perform the later document-level validation/export stage.

**How:** An independent XML text reader applies only Word's text-encoding conventions for tabs and line breaks. Page/column breaks and field instructions are checked as separate events. We check literal text fragments as well as the visible paragraph string, so invisible events or spaces cannot silently disappear.

A small in-memory Word fixture additionally checks an interleaved paragraph/table/paragraph sequence, multiple cell paragraphs, whitespace, formatting, and a page break between two text fragments. It is never written to disk. This tests behavior that the source's regular tables and page-break-only paragraphs do not cover.

In [ ]:
def xml_visible_text(xml):
    pieces = []
    for element in xml.iter():
        if element.tag == W + "t":
            pieces.append(element.text or "")
        elif element.tag == W + "tab":
            pieces.append("\t")
        elif element.tag == W + "cr" or (
            element.tag == W + "br" and element.get(W + "type", "textWrapping") == "textWrapping"):
            pieces.append("\n")
    return "".join(pieces)

def all_paragraph_records(blocks):
    return [p for b in blocks for p in ([b] if b["type"] == "paragraph" else table_paragraphs(b))]

body_paragraph_records = all_paragraph_records(ordered_blocks)
xml_paragraphs = list(body.iter(W + "p"))
for record, xml in zip(body_paragraph_records, xml_paragraphs, strict=True):
    assert record["raw_text"] == xml_visible_text(xml), record["source"]
    assert record["raw_text"] == "".join(r["raw_text"] for r in record["runs"]), record["source"]
    assert [e["text"] for r in record["runs"] for e in r["events"] if e["type"] == "text"] == [
        e.text or "" for e in xml.iter(W + "t")], record["source"]
    assert [(m["xml_tag"], m["kind"], m["clear"]) for m in record["break_markers"]] == [
        (e.tag.rsplit("}", 1)[-1], e.get(W + "type", "textWrapping"), e.get(W + "clear"))
        for e in xml.iter() if e.tag in {W + "br", W + "cr"}], record["source"]

assert [b["position"] for b in ordered_blocks] == [
    i for i, e in enumerate(body) if e.tag in {W + "p", W + "tbl"}]
assert [b["type"] for b in ordered_blocks] == [
    "paragraph" if e.tag == W + "p" else "table" for e in body if e.tag in {W + "p", W + "tbl"}]
assert len(ordered_blocks) + len(layout_markers) == len(body)
all_records = list(ordered_blocks) + [p for b in ordered_blocks if b["type"] == "table" for p in table_paragraphs(b)] + layout_markers
for a in auxiliary_content:
    all_records.extend(a["blocks"])
    all_records.extend(p for b in a["blocks"] if b["type"] == "table" for p in table_paragraphs(b))
    all_records.extend(a["layout_markers"])
assert len({r["id"] for r in all_records}) == len(all_records)
assert all(r["id"] == base_record(r["type"], r["position"], r["source"]["part"],
                                 r["source"]["path"], manifest["sha256"])["id"] for r in all_records)
all_paragraphs = body_paragraph_records + [p for a in auxiliary_content for p in all_paragraph_records(a["blocks"])]
unhandled = [e for p in all_paragraphs for r in p["runs"] for e in r["events"] if e["type"] == "unhandled_xml"]
assert not unhandled, "Unhandled run XML was retained; review before calling extraction complete"
for a in auxiliary_content:
    aux_paragraphs = all_paragraph_records(a["blocks"])
    for record, xml in zip(aux_paragraphs, auxiliary_xml[a["part"]].iter(W + "p"), strict=True):
        assert record["raw_text"] == xml_visible_text(xml)
        assert [e["text"] for r in record["runs"] for e in r["events"] if e["type"] == "field_instruction"] == [
            e.text or "" for e in xml.iter(W + "instrText")]
        assert [e["kind"] for r in record["runs"] for e in r["events"] if e["type"] == "field_char"] == [
            e.get(W + "fldCharType") for e in xml.iter(W + "fldChar")]

summary = {
    "body paragraph records": sum(b["type"] == "paragraph" for b in ordered_blocks),
    "body table records": sum(b["type"] == "table" for b in ordered_blocks),
    "table rows": sum(len(b["rows"]) for b in ordered_blocks if b["type"] == "table"),
    "table cells": sum(len(r["cells"]) for b in ordered_blocks if b["type"] == "table" for r in b["rows"]),
    "all body/cell paragraph records": len(body_paragraph_records),
    "page breaks": sum(m["kind"] == "page" for p in body_paragraph_records for m in p["break_markers"]),
    "empty visible body paragraphs": sum(b["raw_text"] == "" for b in ordered_blocks if b["type"] == "paragraph"),
}
assert list(summary.values())[:6] == [counts[k] for k in (
    "top-level paragraphs", "top-level tables", "table rows", "table cells",
    "all body/table paragraphs", "explicit page breaks")]
show_rows([{"measure": k, "preserved": v} for k, v in summary.items()])
print("PASS: order, source positions, text fragments, whitespace, breaks, IDs, and auxiliary field events")


In [ ]:
from docx.enum.text import WD_BREAK

fixture = Document()
p = fixture.add_paragraph()
p.add_run("  Начало\t ").bold = True
r = p.add_run("before")
r.italic = True
r.add_break(WD_BREAK.PAGE)
r.add_text("after")
r.add_break(WD_BREAK.LINE)
r.add_text("end  ")
t = fixture.add_table(rows=1, cols=2)
t.cell(0, 0).text = " cell 0 "
t.cell(0, 0).add_paragraph("second paragraph")
t.cell(0, 1).text = "Български"
fixture.add_paragraph("")
fixture.add_paragraph("tail")
buffer = BytesIO()
fixture.save(buffer)
fixture_bytes = buffer.getvalue()
fixture_hash = sha256(fixture_bytes).hexdigest()
with ZipFile(BytesIO(fixture_bytes)) as z:
    fixture_body = ET.fromstring(z.read("word/document.xml")).find("w:body", NS)
fixture_parser = Document(BytesIO(fixture_bytes))
fixture_blocks, _ = extract_container(fixture_parser.iter_inner_content(), fixture_body,
                                     "word/document.xml", "/body", fixture_hash)
assert [b["type"] for b in fixture_blocks] == ["paragraph", "table", "paragraph", "paragraph"]
assert fixture_blocks[0]["raw_text"] == "  Начало\t beforeafter\nend  "
assert fixture_blocks[0]["runs"][0]["formatting"]["bold"] is True
assert fixture_blocks[0]["runs"][1]["formatting"]["italic"] is True
assert fixture_blocks[0]["runs"][1]["formatting"]["bold"] is None
assert [e["type"] for e in fixture_blocks[0]["runs"][1]["events"]] == ["text", "break", "text", "break", "text"]
assert [m["kind"] for m in fixture_blocks[0]["break_markers"]] == ["page", "textWrapping"]
assert [p["raw_text"] for p in fixture_blocks[1]["rows"][0]["cells"][0]["paragraphs"]] == [" cell 0 ", "second paragraph"]
assert fixture_blocks[2]["raw_text"] == ""
assert fixture_blocks[3]["raw_text"] == "tail"
fixture_again, _ = extract_container(fixture_parser.iter_inner_content(), fixture_body,
                                    "word/document.xml", "/body", fixture_hash)
assert fixture_again == fixture_blocks
show_rows([preview_record(b) for b in fixture_blocks])
print("PASS: in-memory fixture — interleaving, multiple cell paragraphs, exact whitespace, formatting, inline breaks, empty paragraph, repeatability")


## Step 5 — Identify document boundaries and structural markers

**What:** Use Step 4's `ordered_blocks` to identify source-document spans, distinguish corpus front matter, assign structural regions, and collect candidate legal markers.

**Why:** `Heading1` alone cannot distinguish the corpus title from a source title. Reference-rule excerpts also contain article citations, so text patterns alone are insufficient.

**How:** Create separate annotation dictionaries keyed by existing block IDs. `ordered_blocks`, nested cell paragraphs, footer records, and original styles remain unchanged. A before/after checksum checks that no Step 4 record was modified.

**Boundary evidence:** A nonempty `Heading1` paragraph immediately followed by a non-heading subtitle paragraph and the observed six-row/two-column table labels. We inspect table labels only to recognize the structure; no jurisdiction/date/URL values are interpreted. The corpus heading lacks this pattern.

**Conservative choice:** No fuzzy title matching, page-break-based boundaries, or hardcoded split positions. Unmatched Heading 1 paragraphs after the first verified source raise an explicit review error. A valid source span must contain exactly one exact Heading 2 full-text marker and at most one preceding reference-rules marker. Unexpected layouts or ambiguous markers stop detection instead of being guessed.

**Representation:** `document_boundaries` holds minimal source spans and evidence IDs. `block_annotations` holds membership, region, and observed Word heading level. These are Step 5 annotations, not Step 6's combined corpus representation or a legal hierarchy. Span indices are zero-based indices into `ordered_blocks`; ends are exclusive. Displayed body positions use Step 4's source coordinates.

### 5a. Recognize and audit document starts

Inspect the full Heading 1 audit below. There should be 22 candidates: the corpus title plus 21 verified source starts. IDs are anchored to the source title block, not inferred legal-document identifiers.

In [ ]:
import re
from collections import defaultdict

step4_digest_before = sha256(json.dumps(ordered_blocks, ensure_ascii=False, sort_keys=True).encode("utf-8")).hexdigest()
METADATA_LABELS = (
    "Юрисдикция / Jurisdiction", "Източник / Source URL", "ДВ / Official reference",
    "Версия / Version date", "Достъпено / Accessed", "Бележка / Note",
)
FULL_TEXT_MARKER = "Пълен текст / Full text"
REFERENCE_MARKER = "Извлечени правила (reference) / Extracted rules"

def heading_level(block):
    if block["type"] != "paragraph":
        return None
    match = re.fullmatch(r"Heading([1-9])", block["style"]["resolved_id"] or "")
    return int(match.group(1)) if match else None

def nonempty_paragraph(block):
    return block["type"] == "paragraph" and bool(block["raw_text"]) and not block["raw_text"].isspace()

def matches_metadata_layout(block):
    if block["type"] != "table" or len(block["rows"]) != len(METADATA_LABELS):
        return False
    for row, expected_label in zip(block["rows"], METADATA_LABELS):
        if len(row["cells"]) != 2:
            return False
        label_paragraphs = row["cells"][0]["paragraphs"]
        if len(label_paragraphs) != 1 or label_paragraphs[0]["raw_text"] != expected_label:
            return False
    return True

def discover_document_starts(blocks):
    starts, audit = [], []
    for index, block in enumerate(blocks):
        if heading_level(block) != 1:
            continue
        follows = blocks[index + 1:index + 3]
        subtitle_ok = (len(follows) == 2 and nonempty_paragraph(follows[0])
                       and heading_level(follows[0]) is None)
        table_ok = len(follows) == 2 and matches_metadata_layout(follows[1])
        accepted = nonempty_paragraph(block) and subtitle_ok and table_ok
        audit.append({"index": index, "position": block["position"], "title": block["raw_text"],
                      "subtitle pattern": subtitle_ok, "table-label pattern": table_ok,
                      "verified start": accepted})
        if accepted:
            starts.append(index)
    if not starts:
        raise ValueError("No source starts match the title/subtitle/table pattern; review source structure")
    unmatched = [a["position"] for a in audit if not a["verified start"] and a["index"] >= starts[0]]
    if unmatched:
        raise ValueError(f"Unmatched Heading1 inside source material at body positions {unmatched}; review")
    return starts, audit

source_start_indices, boundary_audit = discover_document_starts(ordered_blocks)
show_rows(boundary_audit)
print("Verified source-document starts:", len(source_start_indices))
assert len(source_start_indices) == 21, "This source snapshot should contain 21 verified sources; review changes"


### 5b. Assign regions using explicit delimiters

| Region | Rule |
|---|---|
| `front_matter` | All blocks before the first verified source title; document ID is `None` |
| `document_metadata` | Source title, subtitle, table, and any following blocks before a region marker |
| `reference_rules` | Exact reference-rules heading through the block before the full-text heading, when present |
| `full_text` | Exact full-text heading through the block before the next verified source title, or end of body |

**Choices:** Each region marker belongs to the region it introduces. A separator/page-break paragraph before the next title remains in the preceding span, unchanged; breaks alone do not change membership. The two documents without reference-rule sections get no invented reference region. Table-cell paragraphs inherit their containing table's membership and region. Footer and layout records remain separate.

Exact headings are required so a sentence that merely quotes “Full text” cannot change the region. Duplicate, missing, or reversed markers raise a review error. Labels identify the corpus structure; they do not establish legal authority, currency, or source reliability.

**Inspect:** The 21-row inventory shows inclusive body positions, region counts, and full-text evidence positions. The region summary includes every body block exactly once.

In [ ]:
def is_region_marker(block, label):
    return heading_level(block) == 2 and block["raw_text"] == label

def label_documents(blocks):
    starts, audit = discover_document_starts(blocks)
    documents, annotations = [], {}
    for block in blocks[:starts[0]]:
        annotations[block["id"]] = {"document_id": None, "region": "front_matter",
                                    "word_heading_level": heading_level(block),
                                    "region_evidence_block_id": None}
    for ordinal, start in enumerate(starts, start=1):
        stop = starts[ordinal] if ordinal < len(starts) else len(blocks)
        full = [i for i in range(start, stop) if is_region_marker(blocks[i], FULL_TEXT_MARKER)]
        refs = [i for i in range(start, stop) if is_region_marker(blocks[i], REFERENCE_MARKER)]
        if len(full) != 1 or len(refs) > 1:
            raise ValueError(f"Review source at body position {blocks[start]['position']}: "
                             f"full-text markers={len(full)}, reference markers={len(refs)}")
        if full[0] <= start + 2 or (refs and not start + 2 < refs[0] < full[0]):
            raise ValueError(f"Region markers out of order at body position {blocks[start]['position']}")
        document_id = "source-document:" + blocks[start]["id"]
        documents.append({"document_id": document_id, "ordinal": ordinal,
            "title": blocks[start]["raw_text"], "start_index": start, "end_index_exclusive": stop,
            "first_body_position": blocks[start]["position"], "last_body_position": blocks[stop - 1]["position"],
            "evidence": {"title_block_id": blocks[start]["id"], "subtitle_block_id": blocks[start + 1]["id"],
                         "metadata_table_block_id": blocks[start + 2]["id"],
                         "reference_marker_block_id": blocks[refs[0]]["id"] if refs else None,
                         "full_text_marker_block_id": blocks[full[0]]["id"]}})
        for i in range(start, stop):
            if i >= full[0]:
                region, evidence_id = "full_text", blocks[full[0]]["id"]
            elif refs and i >= refs[0]:
                region, evidence_id = "reference_rules", blocks[refs[0]]["id"]
            else:
                region, evidence_id = "document_metadata", blocks[start + 2]["id"]
            block = blocks[i]
            annotations[block["id"]] = {"document_id": document_id, "region": region,
                "word_heading_level": heading_level(block), "region_evidence_block_id": evidence_id}
    return documents, annotations

document_boundaries, block_annotations = label_documents(ordered_blocks)
# Cells retain Step 4 IDs; they inherit only the table's annotation, not additional semantic metadata.
cell_paragraph_annotations = {}
for block in ordered_blocks:
    if block["type"] == "table":
        for paragraph in table_paragraphs(block):
            inherited = dict(block_annotations[block["id"]])
            inherited.update(container_block_id=block["id"], word_heading_level=heading_level(paragraph))
            cell_paragraph_annotations[paragraph["id"]] = inherited
block_by_id = {b["id"]: b for b in ordered_blocks}
region_counts = Counter(a["region"] for a in block_annotations.values())
inventory = []
for document in document_boundaries:
    span = ordered_blocks[document["start_index"]:document["end_index_exclusive"]]
    regions = Counter(block_annotations[b["id"]]["region"] for b in span)
    inventory.append({"source": document["ordinal"], "title": document["title"],
        "first body position": document["first_body_position"], "last body position (inclusive)": document["last_body_position"],
        "document_metadata": regions["document_metadata"], "reference_rules": regions["reference_rules"],
        "full_text": regions["full_text"],
        "full-text marker position": block_by_id[document["evidence"]["full_text_marker_block_id"]]["position"]})
show_rows(inventory)
show_rows([{"region": region, "body blocks": count} for region, count in region_counts.items()])
print("Table-cell paragraphs inheriting their table's region:", len(cell_paragraph_annotations))


### 5c. Review complete spans and boundary evidence

**What/how:** First inspect a small window around every verified start, including the preceding block. Then choose a document to view its complete boundary descriptor, all region intervals, the original opening text, and the ending/next-source transition.

The default is source 20, a short document without reference rules. Its expandable block listing includes **every block and complete text** in the span, not just an opening sample. Choose source 1–21 to inspect another document. Descriptors point to original blocks rather than duplicating or rewriting them.

**Inspect:** The corpus title and numbered source list stay in `front_matter`. For the selected source, check that its title/subtitle/table start the metadata region, the full-text heading changes the region once, and the following document title lies outside its exclusive end. Shortened text in the all-boundaries table is display-only.

In [ ]:
def annotated_preview(block, limit=110):
    preview = preview_record(block)
    annotation = block_annotations[block["id"]]
    return {"body position": block["position"], "type": block["type"], "style": block["style"]["resolved_id"],
            "region": annotation["region"], "preview": preview["preview (display only)"][:limit]}

print("Corpus front matter: first five blocks")
show_rows([annotated_preview(b) for b in ordered_blocks[:5]])
windows = []
for document in document_boundaries:
    start = document["start_index"]
    for b in ordered_blocks[max(0, start - 1):start + 4]:
        windows.append({"source start": document["ordinal"], **annotated_preview(b)})
show_rows(windows)

SELECTED_DOCUMENT_ORDINAL = 20  # Change to any integer from 1 to 21.
selected_document = next(d for d in document_boundaries if d["ordinal"] == SELECTED_DOCUMENT_ORDINAL)
start, stop = selected_document["start_index"], selected_document["end_index_exclusive"]
show_full_record(selected_document, "Complete document boundary descriptor and evidence IDs")
intervals = []
for i in range(start, stop):
    b = ordered_blocks[i]
    region = block_annotations[b["id"]]["region"]
    if not intervals or intervals[-1]["region"] != region:
        intervals.append({"region": region, "start index": i, "end index exclusive": i + 1,
                          "first body position": b["position"], "last body position": b["position"]})
    else:
        intervals[-1].update({"end index exclusive": i + 1, "last body position": b["position"]})
show_rows(intervals)
print("Selected source opening:")
show_rows([annotated_preview(b) for b in ordered_blocks[start:start + 5]])
print("Selected source ending and next source:")
show_rows([annotated_preview(b) for b in ordered_blocks[max(start, stop - 3):min(len(ordered_blocks), stop + 3)]])
complete_span_view = []
for b in ordered_blocks[start:stop]:
    entry = {"body_position": b["position"], "block_id": b["id"], "source": b["source"],
             "region": block_annotations[b["id"]]["region"], "type": b["type"], "style": b["style"]}
    if b["type"] == "paragraph":
        entry.update(raw_text=b["raw_text"], break_markers=b["break_markers"])
    else:
        entry["rows"] = [{"row_index": row["row_index"], "cells": [
            {"column_index": cell["column_index"], "paragraphs": [
                {"block_id": p["id"], "raw_text": p["raw_text"]} for p in cell["paragraphs"]]}
            for cell in row["cells"]]} for row in b["rows"]]
    complete_span_view.append(entry)
show_full_record(complete_span_view, "Every block in the selected document span — full text, no truncation")


### 5d. Identify candidate structural/legal markers

**What/how:** Run anchored patterns on full-text paragraphs only. Candidate annotations store the original block ID, source location, exact matched label, character span, rule, and observed Word heading level. Annotations never split paragraphs, merge headings with following titles, or create article spans.

**Conservative rules:**

- Part (`Част`), Chapter (`Глава`), Section (`Раздел`), and Annex (`Приложение`) require both a leading label and an explicit Word heading level 2 or 3. The following label token is retained as written; no numbering conversion is attempted.
- Article (`Чл.` / `Член`) and `§` require a leading numeric identifier, with an optional Cyrillic suffix such as `2а`. They may occur in ordinary paragraphs. Inline mentions and `Членове` are not provision starts.
- Patterns ignore letter case only during matching and tolerate leading spaces/tabs; stored strings are never lowercased, stripped, normalized, or rewritten.
- Reference rules, metadata, front matter, and footers receive no legal-marker candidates. All remain preserved. Exact region-marker headings are delimiters, not legal candidates.

**Trade-off:** Favor inspectable evidence over broad recall. A leading article citation can still be quoted text; it remains a **candidate**, not a confirmed provision. An unstyled chapter label is left unclassified for review. This deliberately avoids guessing a legal hierarchy from Word styles.

**Inspect:** Examples cover every requested marker, both article forms, and repeated/ambiguous headings. Headings with no matching rule remain visible in a review list; they are not discarded.

In [ ]:
# The patterns capture only a leading label/token, not an entire legal provision.
MARKER_RULES = [
    ("part", re.compile(r"^[ \t]*(?P<label>Част\b(?:[ \t]+[^\s.]+)?)", re.IGNORECASE), True),
    ("chapter", re.compile(r"^[ \t]*(?P<label>Глава\b(?:[ \t]+[^\s.]+)?)", re.IGNORECASE), True),
    ("section", re.compile(r"^[ \t]*(?P<label>Раздел\b(?:[ \t]+[^\s.]+)?)", re.IGNORECASE), True),
    ("article", re.compile(r"^[ \t]*(?P<label>(?:Чл\.|Член\b)[ \t]*[0-9]+[а-я]?)(?=$|[\s.,:;()])", re.IGNORECASE), False),
    ("paragraph_sign", re.compile(r"^[ \t]*(?P<label>§[ \t]*[0-9]+[а-я]?)(?=$|[\s.,:;()])", re.IGNORECASE), False),
    ("annex", re.compile(r"^[ \t]*(?P<label>Приложение\b(?:[ \t]+№?[ \t]*(?:[0-9]+[а-я]?|[IVXLCDMІ]+)\b)?)", re.IGNORECASE), True),
]

def detect_legal_candidate(block, annotation):
    if block["type"] != "paragraph" or annotation["region"] != "full_text":
        return None
    if block["raw_text"] in {FULL_TEXT_MARKER, REFERENCE_MARKER}:
        return None
    level = heading_level(block)
    for kind, pattern, requires_heading in MARKER_RULES:
        if requires_heading and level not in {2, 3}:
            continue
        match = pattern.match(block["raw_text"])
        if match:
            return {"kind": kind, "status": "candidate", "block_id": block["id"],
                    "document_id": annotation["document_id"], "source": dict(block["source"]),
                    "matched_label": match.group("label"), "character_span": list(match.span("label")),
                    "word_heading_level": level,
                    "rule": "leading_label_and_heading" if requires_heading else "leading_numbered_label"}
    return None

legal_marker_candidates = [candidate for b in ordered_blocks
    if (candidate := detect_legal_candidate(b, block_annotations[b["id"]])) is not None]
marker_by_block = {m["block_id"]: m for m in legal_marker_candidates}
show_rows([{"candidate kind": kind, "count": count}
           for kind, count in Counter(m["kind"] for m in legal_marker_candidates).items()])
representative_markers = []
for kind in ["part", "chapter", "section", "article", "paragraph_sign", "annex"]:
    representative_markers.append(next(m for m in legal_marker_candidates if m["kind"] == kind))
representative_markers.append(next(m for m in legal_marker_candidates
                                   if m["kind"] == "article" and m["matched_label"].startswith("Член")))
show_rows([{"kind": m["kind"], "body position": block_by_id[m["block_id"]]["position"],
            "matched label (verbatim)": m["matched_label"], "Word heading level": m["word_heading_level"],
            "rule": m["rule"], "source preview": block_by_id[m["block_id"]]["raw_text"][:250]}
           for m in representative_markers])
show_full_record(representative_markers, "Candidate marker annotations with exact evidence locations")

unclassified_headings = [b for b in ordered_blocks
    if block_annotations[b["id"]]["region"] == "full_text" and heading_level(b) in {2, 3}
    and b["raw_text"] != FULL_TEXT_MARKER and b["id"] not in marker_by_block]
print("Full-text headings retained without a candidate classification:", len(unclassified_headings))
show_rows([annotated_preview(b) for b in unclassified_headings[:8]])
heading_occurrences = defaultdict(list)
for b in unclassified_headings:
    heading_occurrences[b["raw_text"]].append(b["position"])
show_rows([{"repeated heading (unchanged)": text, "occurrences": len(positions),
            "first body positions": positions[:8]}
           for text, positions in heading_occurrences.items() if len(positions) > 1][:5])
print("Part and Chapter examples share Heading2: this is Word formatting, not a final legal parent/child relationship.")


### 5e. Check coverage, conservative behavior, and unchanged records

**What/why:** Verify the detected spans partition the body after front matter, each block has exactly one region, and candidate evidence points to unchanged source text. A checksum checks all Step 4 body records, including nested paragraphs and formatting, rather than just preview strings.

**Small focused cases:** A source may omit reference rules; missing/duplicate/out-of-order markers and an unmatched internal Heading 1 must fail. `Чл. 2а.` and `Член 1` should match at the start; inline citations, “Членове,” unstyled chapter labels, and reference-rule excerpts should not. Tests use small copied records in memory and do not alter the source.

These checks concern structural detection only. They do not verify legal meaning or establish a final hierarchy.

In [ ]:
assert len(document_boundaries) == 21
assert len(block_annotations) == len(ordered_blocks)
assert set(block_annotations) == {b["id"] for b in ordered_blocks}
assert sum(region_counts.values()) == len(ordered_blocks)
assert document_boundaries[0]["start_index"] == source_start_indices[0]
assert document_boundaries[-1]["end_index_exclusive"] == len(ordered_blocks)
for left, right in zip(document_boundaries, document_boundaries[1:]):
    assert left["end_index_exclusive"] == right["start_index"]
for d in document_boundaries:
    span = ordered_blocks[d["start_index"]:d["end_index_exclusive"]]
    assert all(block_annotations[b["id"]]["document_id"] == d["document_id"] for b in span)
    assert sum(is_region_marker(b, FULL_TEXT_MARKER) for b in span) == 1
    assert block_annotations[d["evidence"]["full_text_marker_block_id"]]["region"] == "full_text"
    if d["evidence"]["reference_marker_block_id"]:
        assert block_annotations[d["evidence"]["reference_marker_block_id"]]["region"] == "reference_rules"
    assert all(e is None or e in block_by_id for e in d["evidence"].values())
assert all(block_annotations[b["id"]]["document_id"] is None
           for b in ordered_blocks[:source_start_indices[0]])
assert all(a["document_id"] == block_annotations[a["container_block_id"]]["document_id"]
           and a["region"] == block_annotations[a["container_block_id"]]["region"]
           for a in cell_paragraph_annotations.values())
assert len(cell_paragraph_annotations) == len([p for b in ordered_blocks if b["type"] == "table" for p in table_paragraphs(b)])
assert len([d for d in document_boundaries if d["evidence"]["reference_marker_block_id"] is None]) == 2
for marker in legal_marker_candidates:
    b = block_by_id[marker["block_id"]]
    lo, hi = marker["character_span"]
    assert b["raw_text"][lo:hi] == marker["matched_label"]
    assert marker["source"] == b["source"]
    assert block_annotations[b["id"]]["region"] == "full_text"
    assert marker["word_heading_level"] == heading_level(b)
assert sha256(json.dumps(ordered_blocks, ensure_ascii=False, sort_keys=True).encode("utf-8")).hexdigest() == step4_digest_before
print("PASS: 21 contiguous document spans, front matter, exact region coverage, optional references, cell inheritance, marker evidence, unchanged Step 4 records")


In [ ]:
from copy import deepcopy

# Reuse the observed table shape only; synthetic records are isolated from source records.
def test_paragraph(text, style="Normal"):
    return {"id": "unassigned", "position": 0, "type": "paragraph", "raw_text": text,
            "style": {"explicit_id": style, "resolved_id": style, "name": style},
            "source": {"part": "synthetic", "path": "/unassigned"}}

def numbered_test_blocks(blocks):
    result = deepcopy(blocks)
    for i, block in enumerate(result):
        block.update(id=f"test-{i}", position=i, source={"part": "synthetic", "path": f"/children/{i}"})
    return result

sample_doc = [test_paragraph("Test source", "Heading1"), test_paragraph("Subtitle"),
              deepcopy(first_table_record), test_paragraph(FULL_TEXT_MARKER, "Heading2"),
              test_paragraph("Чл. 2а. Exact text.")]
minimal = numbered_test_blocks(sample_doc)
test_documents, test_annotations = label_documents(minimal)
assert len(test_documents) == 1 and test_documents[0]["evidence"]["reference_marker_block_id"] is None
assert test_annotations[minimal[-1]["id"]]["region"] == "full_text"
with_reference = numbered_test_blocks(sample_doc[:3] + [test_paragraph(REFERENCE_MARKER, "Heading2"),
    test_paragraph("чл. 8, т. 1 — reference excerpt")] + sample_doc[3:])
_, reference_annotations = label_documents(with_reference)
assert reference_annotations[with_reference[4]["id"]]["region"] == "reference_rules"

failures = {
    "missing full-text marker": sample_doc[:3] + sample_doc[4:],
    "duplicate full-text marker": sample_doc + [test_paragraph(FULL_TEXT_MARKER, "Heading2")],
    "reversed region markers": sample_doc + [test_paragraph(REFERENCE_MARKER, "Heading2")],
    "duplicate reference marker": sample_doc[:3] + [test_paragraph(REFERENCE_MARKER, "Heading2")]*2 + sample_doc[3:],
    "unmatched internal Heading1": sample_doc + [test_paragraph("Unexpected heading", "Heading1")],
    "marker text with wrong style": sample_doc[:3] + [test_paragraph(FULL_TEXT_MARKER)] + sample_doc[4:],
}
for case, blocks in failures.items():
    try:
        label_documents(numbered_test_blocks(blocks))
    except ValueError:
        pass
    else:
        raise AssertionError(f"Expected an explicit review failure: {case}")

candidate_cases = [
    ("Чл. 2а. Текст", "Normal", "full_text", "article"),
    ("Член 1", "Heading3", "full_text", "article"),
    ("  § 1а. Текст", "Normal", "full_text", "paragraph_sign"),
    ("ГЛАВА II", "Heading2", "full_text", "chapter"),
    ("Раздел ІV", "Heading3", "full_text", "section"),
    ("Приложение № 2а към чл. 7", "Heading2", "full_text", "annex"),
    ("Съгласно чл. 1 се изисква...", "Normal", "full_text", None),
    ("Членове:", "Normal", "full_text", None),
    ("Глава първа.", "Normal", "full_text", None),
    ("Чл. 8. Reference excerpt", "Normal", "reference_rules", None),
    ("Член 1", "Heading3", "document_metadata", None),
]
for text, style, region, expected_kind in candidate_cases:
    block = test_paragraph(text, style)
    result = detect_legal_candidate(block, {"document_id": "test-document", "region": region})
    assert (result["kind"] if result else None) == expected_kind, text
    assert block["raw_text"] == text
print("PASS: optional reference region; six malformed-boundary/marker cases; eleven positive and negative legal-marker cases")


## Step 6 — Assemble the simple intermediate representation

**What:** Build a plain Python `corpus` dictionary from the already-approved Step 4 records and Step 5 annotations. It contains provenance, 21 document descriptors, ordered blocks, auxiliary content, layout markers, and diagnostics.

**Why:** One inspectable representation lets later experiments reuse the same source structure. It does not depend on a chunking strategy, database, or RAG framework.

**How:** Deep-copy the existing records, attach their existing membership and candidate-marker annotations, and point document descriptors to ranges in the ordered block list. No new boundary or marker detection is performed here.

| Component | Contents |
|---|---|
| `schema_version` | Version 1 of this notebook's intermediate record format |
| `source` | Existing archive/member path, DOCX byte size, SHA-256 |
| `parser` | Parser and XML-inspection implementation/version information |
| `documents` | Existing IDs, source title/subtitle, block range, raw metadata-table rows and evidence |
| `blocks` | Ordered Step 4 records plus Step 5 document, region and structural annotations |
| `auxiliary_content` | Existing footer/header records, including field instructions |
| `layout_markers` | Existing Word section-property records |
| `diagnostics` | Existing inventory/region counts, limitations and assembly-check outcomes |

**Design choices:** The body text exists in one ordered block list, with table paragraphs nested in their original cells. Descriptors reference ranges; they do not contain another full copy of each document. Small title/metadata values are repeated for inspection and retain exact block/cell evidence. Deep copies keep the assembled record independent of earlier notebook variables; the extra memory is acceptable for this source. No production classes or package refactor are introduced.

**Scope boundary:** Carry source metadata rows as verbatim labels and values, not normalized fields such as an interpreted effective date. Step 7's provenance classification and metadata interpretation are deferred. UTF-8 JSON is checked in memory; saving/reloading a parsed artifact belongs to Step 8 and is not implemented.

### 6a. Merge existing records and annotations without changing the inputs

Each body/cell paragraph retains its Step 4 ID, source location, raw text, style, runs and break events. `document_id` and `region` come directly from Step 5. `structural_annotations` carries the existing Word heading level, evidence ID, and candidate marker, if any. Front matter keeps `document_id=None`. Footer content stays separate and is not assigned to a source document.

**Inspect later:** The record viewer will show a full paragraph with these additions alongside its original text and formatting.

In [ ]:
def record_digest(value):
    return sha256(json.dumps(value, ensure_ascii=False, sort_keys=True, allow_nan=False).encode("utf-8")).hexdigest()

step5_inputs_before = record_digest([document_boundaries, block_annotations,
                                     cell_paragraph_annotations, legal_marker_candidates])

def attach_existing_annotations(record, annotation):
    # Mutates only the newly copied record, never the original Step 4/5 objects.
    record["document_id"] = annotation["document_id"]
    record["region"] = annotation["region"]
    record["structural_annotations"] = deepcopy({
        key: value for key, value in annotation.items() if key not in {"document_id", "region"}
    })
    marker = marker_by_block.get(record["id"])
    record["structural_annotations"]["legal_marker_candidates"] = [deepcopy(marker)] if marker else []

assembled_blocks = deepcopy(ordered_blocks)
for assembled in assembled_blocks:
    attach_existing_annotations(assembled, block_annotations[assembled["id"]])
    if assembled["type"] == "table":
        for paragraph in table_paragraphs(assembled):
            attach_existing_annotations(paragraph, cell_paragraph_annotations[paragraph["id"]])

print("Assembled body blocks:", len(assembled_blocks))
print("Original IDs and order retained:", [b["id"] for b in assembled_blocks] == [b["id"] for b in ordered_blocks])


### 6b. Build complete document descriptors with source evidence

**What/how:** Copy each verified boundary's ID, ordinal and evidence. Include title and subtitle exactly as written, and retain the zero-based `[start_index, end_index_exclusive)` range into `corpus['blocks']`. Source body positions remain separate from list indices.

For metadata, each table row keeps its row/column coordinates and separate label/value paragraph lists. Each paragraph includes its exact text, ID and source locator. This avoids joining paragraphs, changing date formats, interpreting jurisdiction strings, or mapping labels to a new metadata vocabulary.

**Trade-off:** Raw rows are less convenient for querying than typed metadata, but they preserve what the source actually supplied and leave Step 7's decisions open. Missing reference-rule sections remain `None` in the existing evidence; nothing is inferred.

**Inspect:** The first document descriptor is displayed in full below, including all six metadata rows and their evidence.

In [ ]:
def text_with_evidence(paragraph):
    return {"raw_text": paragraph["raw_text"], "block_id": paragraph["id"],
            "source": deepcopy(paragraph["source"])}

assembled_documents = []
for boundary in document_boundaries:
    evidence = boundary["evidence"]
    title = block_by_id[evidence["title_block_id"]]
    subtitle = block_by_id[evidence["subtitle_block_id"]]
    metadata_table = block_by_id[evidence["metadata_table_block_id"]]
    raw_rows = []
    for row in metadata_table["rows"]:
        label_cell, value_cell = row["cells"]  # The six-by-two layout was verified in Step 5.
        raw_rows.append({"row_index": row["row_index"],
            "label_column_index": label_cell["column_index"],
            "value_column_index": value_cell["column_index"],
            "label_paragraphs": [text_with_evidence(p) for p in label_cell["paragraphs"]],
            "value_paragraphs": [text_with_evidence(p) for p in value_cell["paragraphs"]]})
    assembled_documents.append({
        "document_id": boundary["document_id"], "ordinal": boundary["ordinal"],
        "source_titles": {"title": text_with_evidence(title), "subtitle": text_with_evidence(subtitle)},
        "block_range": {key: boundary[key] for key in (
            "start_index", "end_index_exclusive", "first_body_position", "last_body_position")},
        "source_metadata": {"table_block_id": metadata_table["id"],
                            "table_source": deepcopy(metadata_table["source"]), "rows": raw_rows},
        "boundary_evidence": deepcopy(evidence),
    })
print("Document descriptors:", len(assembled_documents))
print("Metadata representation: verbatim table rows with paragraph/cell evidence; no inferred fields")


### 6c. Assemble the corpus envelope

**What/how:** Combine the documents and blocks with the existing source manifest, exact installed parser versions, auxiliary/layout records and diagnostic counts. No timestamp is invented and no package metadata or external source is newly read.

The diagnostic notes describe limitations already visible in earlier steps: candidate markers remain provisional, some Word headings are unclassified, and a footer field does not supply rendered page numbers. They do not imply that legal content has been verified.

**Inspect:** The overview shows keys and collection sizes, not a dump of the whole corpus. A document descriptor references `blocks[start:end]`; table cells remain nested under a table in that sequence.

In [ ]:
corpus = {
    "schema_version": 1,
    "source": deepcopy(manifest),
    "parser": {"name": "python-docx", "version": version("python-docx"),
               "lxml_version": version("lxml"), "python_version": sys.version.split()[0],
               "xml_inspector": "xml.etree.ElementTree"},
    "documents": assembled_documents,
    "blocks": assembled_blocks,
    "auxiliary_content": deepcopy(auxiliary_content),
    "layout_markers": deepcopy(layout_markers),
    "diagnostics": {
        "inventory_counts": deepcopy(counts),
        "region_counts": dict(region_counts),
        "candidate_marker_counts": dict(Counter(m["kind"] for m in legal_marker_candidates)),
        "warnings": [
            {"code": "candidate_markers_only", "count": len(legal_marker_candidates),
             "message": "Legal-marker annotations remain candidates; no final legal hierarchy exists."},
            {"code": "unclassified_headings", "count": len(unclassified_headings),
             "message": "Unclassified full-text headings are preserved without assigning a legal role."},
            {"code": "rendered_page_numbers_unavailable",
             "message": "Explicit breaks and footer field instructions are retained; rendered page numbers are unavailable."},
        ],
        "assembly_checks": {},
    },
}
show_rows([{"component": key, "representation": type(value).__name__,
            "entries": len(value) if isinstance(value, (dict, list)) else value}
           for key, value in corpus.items()])
show_full_record({"schema_version": corpus["schema_version"], "source": corpus["source"],
                  "parser": corpus["parser"]}, "Corpus provenance and parser information")


### 6d. Inspect representative assembled records

The first display is one **complete document descriptor**, including raw metadata and every evidence reference. Expand the table and paragraph displays to see full records rather than previews.

The contiguous sequence shows the title, subtitle, metadata table, and following reference-region heading. Their shared document ID connects them to the descriptor; their regions retain the Step 5 distinctions. An article paragraph demonstrates how a candidate marker attaches to a complete original paragraph without splitting it. A separate footer view shows its field events.

Only the short sequence table truncates displayed text. The expandable JSON records contain the complete selected values.

In [ ]:
INSPECT_DOCUMENT_ORDINAL = 1
ir_document = next(d for d in corpus["documents"] if d["ordinal"] == INSPECT_DOCUMENT_ORDINAL)
ir_by_id = {b["id"]: b for b in corpus["blocks"]}
ir_table = ir_by_id[ir_document["source_metadata"]["table_block_id"]]
ir_article = next(b for b in corpus["blocks"] if b["document_id"] == ir_document["document_id"]
                  and b["type"] == "paragraph"
                  and any(m["kind"] == "article" for m in b["structural_annotations"]["legal_marker_candidates"]))

# Direct display shows the descriptor immediately; expandable views keep long records navigable.
display(ir_document)
show_full_record(ir_document, "Complete document descriptor — all titles, metadata rows, range and evidence")
show_full_record(ir_article, "Complete representative article paragraph — original text, runs and candidate annotation")
show_full_record(ir_table, "Complete representative table — nested annotated cell paragraphs")
sequence_start = ir_document["block_range"]["start_index"]
show_rows([{"index in corpus blocks": i, "body position": b["position"], "type": b["type"],
            "document ID": b["document_id"], "region": b["region"],
            "style": b["style"]["resolved_id"], "text preview": preview_record(b)["preview (display only)"]}
           for i, b in enumerate(corpus["blocks"][sequence_start:sequence_start + 4], start=sequence_start)])
show_full_record(corpus["auxiliary_content"][0], "Auxiliary footer retained separately with field instructions")


### 6e. Check assembly integrity and JSON compatibility

**What/why:** Check that assembly only adds the approved annotations: IDs/order, every original field, document ranges, metadata evidence, candidate markers and auxiliary content must still agree with Steps 4–5. These are local assembly checks, not RAG evaluation or Step 8's full parsing validation/export stage.

**How:** Compare original fields to the assembled copies and resolve every descriptor's evidence references. Check the earlier inputs' digests again. Finally encode the corpus with `json.dumps(..., ensure_ascii=False, allow_nan=False)` as UTF-8 **in memory**, displaying only the byte count. No JSON file is written and no saved-artifact round trip is performed.

**Choice:** Ordinary dictionaries/lists are sufficient at this learning stage. The JSON check catches unsupported Python objects while preserving readable Cyrillic. Version 1 describes this intermediate interface, not a finalized legal-data schema.

In [ ]:
def assert_original_fields_preserved(original, assembled):
    # Every original field survives; added annotations are checked separately.
    for key, value in original.items():
        if key != "rows":
            assert assembled[key] == value, (original["id"], key)
    assert assembled is not original
    if original["type"] == "table":
        assert len(assembled["rows"]) == len(original["rows"])
        for row_before, row_after in zip(original["rows"], assembled["rows"], strict=True):
            assert row_after["row_index"] == row_before["row_index"]
            for cell_before, cell_after in zip(row_before["cells"], row_after["cells"], strict=True):
                assert cell_after["column_index"] == cell_before["column_index"]
                for before, after in zip(cell_before["paragraphs"], cell_after["paragraphs"], strict=True):
                    assert_original_fields_preserved(before, after)

assert corpus["source"] == manifest
assert corpus["source"] is not manifest
assert len(corpus["documents"]) == len(document_boundaries) == 21
for original, assembled in zip(ordered_blocks, corpus["blocks"], strict=True):
    assert_original_fields_preserved(original, assembled)

paragraphs_and_tables = list(corpus["blocks"]) + [p for b in corpus["blocks"] if b["type"] == "table" for p in table_paragraphs(b)]
all_ir_records = {b["id"]: b for b in paragraphs_and_tables}
assert len(all_ir_records) == len(paragraphs_and_tables)
for assembled in paragraphs_and_tables:
    annotation = (block_annotations if assembled["id"] in block_annotations else cell_paragraph_annotations)[assembled["id"]]
    assert assembled["document_id"] == annotation["document_id"]
    assert assembled["region"] == annotation["region"]
    structural = assembled["structural_annotations"]
    assert {k: v for k, v in structural.items() if k != "legal_marker_candidates"} == {
        k: v for k, v in annotation.items() if k not in {"document_id", "region"}}
    expected_marker = marker_by_block.get(assembled["id"])
    assert structural["legal_marker_candidates"] == ([expected_marker] if expected_marker else [])

for descriptor, boundary in zip(corpus["documents"], document_boundaries, strict=True):
    assert descriptor["document_id"] == boundary["document_id"]
    assert descriptor["boundary_evidence"] == boundary["evidence"]
    assert descriptor["block_range"] == {k: boundary[k] for k in descriptor["block_range"]}
    span = corpus["blocks"][descriptor["block_range"]["start_index"]:descriptor["block_range"]["end_index_exclusive"]]
    assert span and all(b["document_id"] == descriptor["document_id"] for b in span)
    for title in descriptor["source_titles"].values():
        evidence = all_ir_records[title["block_id"]]
        assert title["raw_text"] == evidence["raw_text"] and title["source"] == evidence["source"]
    metadata = descriptor["source_metadata"]
    table = all_ir_records[metadata["table_block_id"]]
    assert metadata["table_source"] == table["source"]
    for metadata_row, row in zip(metadata["rows"], table["rows"], strict=True):
        assert metadata_row["row_index"] == row["row_index"]
        for field, column in [("label_paragraphs", 0), ("value_paragraphs", 1)]:
            assert metadata_row[field] == [text_with_evidence(p) for p in row["cells"][column]["paragraphs"]]
    assert all(value is None or value in all_ir_records for value in descriptor["boundary_evidence"].values())

assert corpus["auxiliary_content"] == auxiliary_content and corpus["auxiliary_content"] is not auxiliary_content
assert corpus["layout_markers"] == layout_markers and corpus["layout_markers"] is not layout_markers
assert record_digest(ordered_blocks) == step4_digest_before
assert record_digest([document_boundaries, block_annotations, cell_paragraph_annotations,
                      legal_marker_candidates]) == step5_inputs_before
corpus["diagnostics"]["assembly_checks"] = {
    "original_fields_and_order": "passed", "document_ranges_and_membership": "passed",
    "metadata_and_title_evidence": "passed", "annotations_and_candidates": "passed",
    "auxiliary_content_and_layout": "passed", "step4_step5_inputs_unchanged": "passed",
}
# No filesystem output. The temporary string/bytes are discarded after measuring their size.
json_utf8_size = len(json.dumps(corpus, ensure_ascii=False, allow_nan=False).encode("utf-8"))
show_rows([{"assembly check": check, "result": result}
           for check, result in corpus["diagnostics"]["assembly_checks"].items()])
print("PASS: corpus is UTF-8 JSON serializable; in-memory encoded size:", json_utf8_size, "bytes")
print("PASS: 21 descriptors,", len(corpus["blocks"]), "ordered body records, unchanged source text/provenance, resolved evidence")


## Step 7 — Separate extracted, generated, structurally interpreted, and deferred metadata

**What:** Add an in-memory `corpus['metadata_provenance']` catalog that classifies existing metadata without replacing Step 6's source records. The catalog covers all 21 documents; the example uses the **Black Sea Coast Spatial Development Act**.

**Why:** A source's “Version date” string is not a legal effective-date conclusion. A source heading is different from our candidate interpretation of it. A file hash or record ID is generated by ingestion, even though it helps locate source evidence.

| Category | Meaning in this notebook |
|---|---|
| `source_extracted` | Verbatim source titles, metadata-table labels/values, explicit marker text/styles, and separately scoped DOCX package properties |
| `generated` | File hash, measured byte size, local archive/member locator, IDs, encounter ordinal, positions/counts, parser/version and schema version |
| `structurally_interpreted` | Existing detected boundaries, membership, region assignments, resolved heading levels and candidate legal-marker roles |
| `deferred` | Decisions/fields not computed: normalized legal hierarchy, legal-date interpretation, language/topic/property inference, chunk metadata and its inheritance policy |

**How:** Build categorized views from the assembled `corpus`. Add friendly field keys by exact bilingual label lookup; preserve each original label and every value paragraph, with its table/cell/paragraph evidence. No values are stripped, normalized, translated, parsed as dates, repaired, reconciled, or filled in. Unknown or duplicate source rows would remain separate entries, not be overwritten.

**Important distinctions:** The container name `corpus['source']` means ingestion provenance; its local path/hash/byte count are not claims extracted from a legal source. A source URL inside a document's table *is* source-extracted metadata. Evidence locations are code-generated addresses pointing to unchanged source values.

The catalog is additive. Existing Step 6 fields stay unchanged. No metadata is copied into chunks, and **which document fields chunks should inherit remains undecided**. Existing table-cell membership is structural bookkeeping, not a chunk inheritance policy.

### 7a. Preserve raw metadata fields with evidence

**What/how:** Carry all six table fields — Jurisdiction, Source URL, Official Reference, Version Date, Accessed, Note — and titles into `source_extracted`. Friendly keys such as `version_date` classify labels only; the raw date strings remain authoritative source values for this stage. Lists preserve paragraph boundaries rather than joining them into new text.

**Trade-off:** Strings and evidence lists are less convenient than normalized typed fields, but avoid conflating the source's claims with our interpretation. Missing/empty values remain as supplied. Legal currency and validity are not verified here.

In [ ]:
# Snapshot only the existing Step 6 fields; supports rerunning Step 7 cells in the same kernel.
step6_field_names = [key for key in corpus if key != "metadata_provenance"]
step6_before_metadata = record_digest({key: corpus[key] for key in step6_field_names})
FIELD_BY_SOURCE_LABEL = {
    "Юрисдикция / Jurisdiction": "jurisdiction",
    "Източник / Source URL": "source_url",
    "ДВ / Official reference": "official_reference",
    "Версия / Version date": "version_date",
    "Достъпено / Accessed": "accessed",
    "Бележка / Note": "note",
}

def classified_source_rows(descriptor):
    metadata = descriptor["source_metadata"]
    fields = []
    for row in metadata["rows"]:
        labels = [p["raw_text"] for p in row["label_paragraphs"]]
        # No fuzzy matching or normalization. Keep unfamiliar/multi-paragraph labels verbatim.
        field_key = FIELD_BY_SOURCE_LABEL.get(labels[0]) if len(labels) == 1 else None
        fields.append({"field": field_key, "raw_labels": labels,
            "raw_values": [p["raw_text"] for p in row["value_paragraphs"]],
            "evidence": {"table_block_id": metadata["table_block_id"],
                         "table_source": deepcopy(metadata["table_source"]),
                         "row_index": row["row_index"],
                         "label_column_index": row["label_column_index"],
                         "value_column_index": row["value_column_index"],
                         "label_paragraphs": deepcopy(row["label_paragraphs"]),
                         "value_paragraphs": deepcopy(row["value_paragraphs"])}})
    return fields

metadata_blocks_by_id = {b["id"]: b for b in corpus["blocks"]}
metadata_documents = []
for descriptor in corpus["documents"]:
    start = descriptor["block_range"]["start_index"]
    end = descriptor["block_range"]["end_index_exclusive"]
    span = corpus["blocks"][start:end]
    explicit_markers = []
    for key in ["reference_marker_block_id", "full_text_marker_block_id"]:
        marker_id = descriptor["boundary_evidence"][key]
        if marker_id is not None:
            marker_block = metadata_blocks_by_id[marker_id]
            explicit_markers.append({**text_with_evidence(marker_block),
                                     "explicit_style_id": marker_block["style"]["explicit_id"]})
    regions = []
    for region in dict.fromkeys(b["region"] for b in span):
        region_blocks = [b for b in span if b["region"] == region]
        regions.append({"region": region,
            "evidence_block_ids": list(dict.fromkeys(
                b["structural_annotations"]["region_evidence_block_id"] for b in region_blocks))})
    candidates = [deepcopy(m) for b in span for m in b["structural_annotations"]["legal_marker_candidates"]]
    metadata_documents.append({
        "document_id": descriptor["document_id"],
        "source_extracted": {"titles": deepcopy(descriptor["source_titles"]),
                             "metadata_fields": classified_source_rows(descriptor),
                             "explicit_region_marker_texts": explicit_markers},
        "generated": {"document_id": descriptor["document_id"], "encounter_ordinal": descriptor["ordinal"],
                      "body_block_count": len(span),
                      "first_body_position": descriptor["block_range"]["first_body_position"],
                      "last_body_position": descriptor["block_range"]["last_body_position"]},
        "structurally_interpreted": {
            "block_range": deepcopy(descriptor["block_range"]),
            "boundary_evidence": deepcopy(descriptor["boundary_evidence"]),
            "regions": regions, "candidate_legal_markers": candidates,
        },
    })
print("Classified document metadata for", len(metadata_documents), "documents")
print("Source metadata rows retained:", sum(len(d["source_extracted"]["metadata_fields"]) for d in metadata_documents))


### 7b. Keep package properties separate from legal-document metadata

**What/how:** Read `docProps/core.xml` from the same in-memory DOCX bytes loaded in Step 1. Preserve every property's exact text value, qualified XML name, attributes, and element location. Empty elements retain `None` as their XML text value; they are not filled from other documents.

**Why:** The package creator is `python-docx`, and its created/modified strings are `2013-12-23T23:15:00Z`. These values were already in the supplied DOCX. They are **source-extracted package properties**, not timestamps generated by our ingestion, not legal publication/version/effective dates, and not evidence of current legal validity.

No source timestamp is copied into an ingestion timestamp. No current ingestion timestamp is invented. No effort is made to reproduce reference-chunk dates or hours.

**Inspect:** The package-property table below is corpus-scoped; these fields are not attached to each legal document or any future chunks.

In [ ]:
CORE_PROPERTIES_PART = "docProps/core.xml"
with ZipFile(BytesIO(docx_bytes)) as metadata_package:
    core_properties_root = ET.fromstring(metadata_package.read(CORE_PROPERTIES_PART))
package_properties = []
for index, element in enumerate(core_properties_root):
    package_properties.append({"qualified_name": element.tag,
        "name": element.tag.rsplit("}", 1)[-1], "raw_value": element.text,
        "attributes": dict(element.attrib),
        "source": {"part": CORE_PROPERTIES_PART, "path": f"/coreProperties/children/{index}"}})
show_rows([{"property": p["name"], "raw value": p["raw_value"],
            "category": "source_extracted — DOCX package only",
            "evidence": p["source"]["part"] + ":" + p["source"]["path"]}
           for p in package_properties])


### 7c. Make generated, interpreted, and deferred fields explicit

**Generated:** Record IDs, positions and counts are computed, even when computed deterministically. The ordinal is the parser's encounter order. The measured file path/bytes/hash and software versions describe the ingestion run, not the legal source. No generated timestamp is required.

**Structurally interpreted:** Choosing which blocks belong to a document or region, resolving Word heading levels, and identifying a candidate article/annex are code decisions with source evidence. Candidate roles remain provisional. The original heading text/style is source-extracted; its interpreted role is a separate field.

**Deferred:** List future fields as *not computed*, without placeholder values that could be mistaken for extracted facts. Do not decide chunk inheritance, which metadata belongs on chunks, or how parent/child chunks should propagate metadata.

The field guide maps existing corpus paths to their provenance categories. It describes existing fields; it neither rewrites them nor assigns a final legal hierarchy.

In [ ]:
DEFERRED_METADATA = [
    {"field": "normalized_legal_hierarchy", "status": "deferred", "reason": "Candidate markers are not a final legal tree."},
    {"field": "legal_date_interpretation", "status": "deferred", "reason": "Effective dates, amendment applicability and legal currency require later interpretation; source date strings stay unchanged."},
    {"field": "language_classification", "status": "deferred", "reason": "No language detection or inferred language labels now."},
    {"field": "topic_labels", "status": "deferred", "reason": "No inferred topic or domain labels now."},
    {"field": "property_facts", "status": "deferred", "reason": "No extraction or inference of property/investment facts now."},
    {"field": "normalized_jurisdiction_and_source_fields", "status": "deferred", "reason": "Keep raw jurisdiction, URL, reference and date values; no reconciliation or normalization."},
    {"field": "chunk_level_metadata", "status": "deferred", "reason": "Chunks do not exist in this stage."},
    {"field": "document_to_chunk_metadata_inheritance", "status": "deferred", "reason": "Which document fields to copy or inherit is explicitly undecided until chunking/metadata design."},
]
METADATA_FIELD_GUIDE = [
    {"field_path": "documents[*].source_titles / source_metadata.rows", "category": "source_extracted", "note": "Verbatim source values and evidence, not verified legal facts."},
    {"field_path": "blocks[*].style.explicit_id / raw_text / run events", "category": "source_extracted", "note": "Source-authored text, applied style and explicit markers; text is not itself a derived legal role."},
    {"field_path": "auxiliary_content[*].blocks[*].runs[*].events", "category": "source_extracted", "note": "Original footer/header text and field instructions, not rendered page numbers."},
    {"field_path": "metadata_provenance.source_extracted.package_properties", "category": "source_extracted", "note": "Package-level properties only; never substituted for legal dates."},
    {"field_path": "source / parser / schema_version", "category": "generated", "note": "Local locator, measured bytes, hash and software provenance; source URLs inside tables are a different category."},
    {"field_path": "documents[*].document_id / ordinal; blocks[*].id / position / source", "category": "generated", "note": "Generated identity, encounter order and evidence addresses."},
    {"field_path": "diagnostics.*_counts / assembly_checks", "category": "generated", "note": "Computed counts/check results; some counts summarize structurally interpreted labels."},
    {"field_path": "documents[*].block_range / boundary_evidence; blocks[*].document_id / region", "category": "structurally_interpreted", "note": "Membership and regions are inferred from the observed structural pattern; the ID value itself is generated."},
    {"field_path": "blocks[*].style.resolved_id / structural_annotations.word_heading_level", "category": "structurally_interpreted", "note": "Default style resolution and numeric level interpretation do not establish a legal hierarchy."},
    {"field_path": "blocks[*].structural_annotations.legal_marker_candidates", "category": "structurally_interpreted", "note": "Candidate marker roles with exact matched source labels and evidence spans."},
]
metadata_provenance = {
    "source_extracted": {"package_properties": package_properties},
    "generated": {"source_locator_and_fingerprint": deepcopy(corpus["source"]),
                  "parser": deepcopy(corpus["parser"]), "schema_version": corpus["schema_version"],
                  "inventory_counts": deepcopy(corpus["diagnostics"]["inventory_counts"])},
    "documents": metadata_documents,
    "deferred": deepcopy(DEFERRED_METADATA),
    "field_guide": deepcopy(METADATA_FIELD_GUIDE),
}
# Additive catalog: no replacement or mutation of the existing document/block records.
corpus["metadata_provenance"] = metadata_provenance
show_rows(METADATA_FIELD_GUIDE)
show_rows(DEFERRED_METADATA)


### 7d. Actual document example — Black Sea Coast Spatial Development Act

**Ordinal clarification:** In this supplied corpus, the named Act is **Document #9**, not #10. Document #10 is the Local Taxes and Fees Act (Tourist tax section). Select by the exact source subtitle so the example follows the Act you requested rather than the conflicting number.

The four displays show: (1) exact source fields with evidence; (2) generated identity/software/file information; (3) structural decisions and their evidence; (4) deferred fields with no inferred values. Source metadata text is shown in full, including the entire Note.

The structural examples reuse Step 5/6 annotations; no new legal interpretation is performed. Expand the full categorized record for all evidence and candidate annotations. No chunk inheritance policy is selected.

In [ ]:
REQUESTED_SOURCE_SUBTITLE = "Black Sea Coast Spatial Development Act"
matching_documents = [d for d in corpus["documents"]
                      if d["source_titles"]["subtitle"]["raw_text"] == REQUESTED_SOURCE_SUBTITLE]
if len(matching_documents) != 1:
    raise ValueError("Expected exactly one named Act; review the document inventory")
metadata_example_document = matching_documents[0]
metadata_example = next(d for d in metadata_provenance["documents"]
                        if d["document_id"] == metadata_example_document["document_id"])
print("Selected:", metadata_example_document["source_titles"]["subtitle"]["raw_text"],
      "— actual source ordinal", metadata_example_document["ordinal"])
show_rows([{"actual ordinal": d["ordinal"], "source title": d["source_titles"]["title"]["raw_text"],
            "source subtitle": d["source_titles"]["subtitle"]["raw_text"]}
           for d in corpus["documents"] if d["ordinal"] in {metadata_example_document["ordinal"], 10}])

def paragraph_evidence_display(paragraphs):
    return "\n".join(p["source"]["part"] + ":" + p["source"]["path"] for p in paragraphs)

print("1. Source/extracted metadata — values unchanged")
source_display = [
    {"field": key, "original label": "source title paragraph", "raw value": title["raw_text"],
     "category": "source_extracted", "evidence": title["source"]["part"] + ":" + title["source"]["path"]}
    for key, title in metadata_example["source_extracted"]["titles"].items()
]
source_display += [
    {"field": field["field"], "original label": "\n".join(field["raw_labels"]),
     "raw value": "\n".join(field["raw_values"]), "category": "source_extracted",
     "evidence": paragraph_evidence_display(field["evidence"]["value_paragraphs"])}
    for field in metadata_example["source_extracted"]["metadata_fields"]
]
# Newlines here separate display paragraphs only. Stored raw_values remain separate unchanged strings.
show_rows(source_display)
print("2. Generated ingestion metadata — not source legal assertions")
generated_display = [{"field": key, "value": value, "category": "generated", "basis": "existing ingestion record"}
                     for key, value in metadata_example["generated"].items()]
generated_display += [{"field": "file." + key, "value": value, "category": "generated",
                       "basis": "local source locator or measured fingerprint"}
                      for key, value in metadata_provenance["generated"]["source_locator_and_fingerprint"].items()]
generated_display += [
    {"field": "parser", "value": json.dumps(metadata_provenance["generated"]["parser"], ensure_ascii=False),
     "category": "generated", "basis": "installed software"},
    {"field": "schema_version", "value": metadata_provenance["generated"]["schema_version"],
     "category": "generated", "basis": "notebook record format"},
]
show_rows(generated_display)
print("3. Structurally interpreted metadata — evidence-backed code decisions")
interpreted = metadata_example["structurally_interpreted"]
interpreted_display = [{"field": "document boundary range", "value": json.dumps(interpreted["block_range"]),
                       "category": "structurally_interpreted",
                       "evidence": json.dumps(interpreted["boundary_evidence"], ensure_ascii=False)}]
interpreted_display += [{"field": "region", "value": region["region"],
                        "category": "structurally_interpreted", "evidence": ", ".join(region["evidence_block_ids"])}
                       for region in interpreted["regions"]]
example_markers = []
for marker_kind in dict.fromkeys(m["kind"] for m in interpreted["candidate_legal_markers"]):
    example_markers.append(next(m for m in interpreted["candidate_legal_markers"] if m["kind"] == marker_kind))
interpreted_display += [{"field": "candidate legal role", "value": m["kind"] + " (candidate); source label: " + m["matched_label"],
                        "category": "structurally_interpreted",
                        "evidence": m["source"]["part"] + ":" + m["source"]["path"] + " span=" + str(m["character_span"])}
                       for m in example_markers]
show_rows(interpreted_display)
print("4. Deferred metadata — no values computed or inheritance policy selected")
show_rows([{**item, "category": "deferred"} for item in metadata_provenance["deferred"]])
show_full_record(metadata_example, "Complete categorized metadata for the named Act, including all source evidence")


### 7e. Check metadata fidelity and category separation

**What/how:** Compare all 126 metadata rows across all 21 documents with the unchanged Step 6 descriptor rows and actual table paragraph records. Check title, label, value, cell coordinate and evidence fidelity. Compare package properties directly to XML text/attributes. Confirm deferred entries contain decisions/reasons only, without computed values or chunk inheritance choices.

A before/after digest checks every pre-existing Step 6 field. Small synthetic metadata rows additionally check that unknown/duplicate labels and an empty value remain intact. These are focused checks of Step 7 bookkeeping, not Step 8's full validation/export workflow or evaluation.

**Inspect:** A compact pass summary below. Source “Version date” and “Accessed” remain strings; package timestamps remain separately scoped raw properties.

In [ ]:
all_metadata_paragraphs = {p["id"]: p for b in corpus["blocks"]
    for p in ([b] if b["type"] == "paragraph" else table_paragraphs(b))}
for descriptor, classified in zip(corpus["documents"], metadata_provenance["documents"], strict=True):
    assert classified["document_id"] == descriptor["document_id"]
    assert classified["source_extracted"]["titles"] == descriptor["source_titles"]
    fields = classified["source_extracted"]["metadata_fields"]
    assert [f["field"] for f in fields] == list(FIELD_BY_SOURCE_LABEL.values())
    for original_row, field in zip(descriptor["source_metadata"]["rows"], fields, strict=True):
        assert field["raw_labels"] == [p["raw_text"] for p in original_row["label_paragraphs"]]
        assert field["raw_values"] == [p["raw_text"] for p in original_row["value_paragraphs"]]
        evidence = field["evidence"]
        assert evidence["table_block_id"] == descriptor["source_metadata"]["table_block_id"]
        assert evidence["table_source"] == descriptor["source_metadata"]["table_source"]
        for key in ["row_index", "label_column_index", "value_column_index", "label_paragraphs", "value_paragraphs"]:
            assert evidence[key] == original_row[key]
        for item in evidence["label_paragraphs"] + evidence["value_paragraphs"]:
            actual = all_metadata_paragraphs[item["block_id"]]
            assert item["raw_text"] == actual["raw_text"] and item["source"] == actual["source"]
    structural = classified["structurally_interpreted"]
    assert structural["block_range"] == descriptor["block_range"]
    assert structural["boundary_evidence"] == descriptor["boundary_evidence"]
    span = corpus["blocks"][descriptor["block_range"]["start_index"]:descriptor["block_range"]["end_index_exclusive"]]
    assert structural["candidate_legal_markers"] == [m for b in span for m in b["structural_annotations"]["legal_marker_candidates"]]
    for marker in classified["source_extracted"]["explicit_region_marker_texts"]:
        block = metadata_blocks_by_id[marker["block_id"]]
        assert marker["raw_text"] == block["raw_text"] and marker["source"] == block["source"]
        assert marker["explicit_style_id"] == block["style"]["explicit_id"]
for record, element in zip(metadata_provenance["source_extracted"]["package_properties"], core_properties_root, strict=True):
    assert record["raw_value"] == element.text and record["attributes"] == dict(element.attrib)
    assert record["qualified_name"] == element.tag
assert all(item["status"] == "deferred" and set(item) == {"field", "status", "reason"}
           for item in metadata_provenance["deferred"])
assert any(item["field"] == "document_to_chunk_metadata_inheritance" for item in metadata_provenance["deferred"])
assert record_digest({key: corpus[key] for key in step6_field_names}) == step6_before_metadata

# Pure metadata bookkeeping cases: preserve unknown labels, duplicate rows, and empty value text.
synthetic_descriptor = deepcopy(corpus["documents"][0])
synthetic_row = synthetic_descriptor["source_metadata"]["rows"][0]
synthetic_row["label_paragraphs"][0]["raw_text"] = "  Unknown source label  "
synthetic_row["value_paragraphs"][0]["raw_text"] = ""
synthetic_descriptor["source_metadata"]["rows"] = [synthetic_row, deepcopy(synthetic_row)]
synthetic_before = record_digest(synthetic_descriptor)
synthetic_classification = classified_source_rows(synthetic_descriptor)
assert len(synthetic_classification) == 2
assert all(f["field"] is None and f["raw_labels"] == ["  Unknown source label  "] and f["raw_values"] == [""]
           for f in synthetic_classification)
assert record_digest(synthetic_descriptor) == synthetic_before
print("PASS: all 21 documents / 126 source metadata rows and evidence preserved unchanged")
print("PASS: package properties scoped separately; deferred fields contain no inferred values or inheritance decisions")
print("PASS: original Step 6 corpus fields unchanged; unknown/duplicate/empty metadata preserved")


## Step 8 — Validate parsing fidelity and persist Stage 1

**What:** Check the assembled `corpus` against the supplied DOCX and save `data/processed/01_parsed_docx_structure.json` only when required checks pass.

**Why:** Counts alone cannot detect reordered paragraphs, altered text, or metadata detached from its source cells. We compare actual locations and content, while keeping XML-level checks bounded to the structures this file uses.

**How:** Re-read the same archive member to verify its identity. Compare body/cell paragraphs and table order against XML; reconcile boundaries, regions, metadata, IDs, footer fields and marker evidence. Each check produces a named PASS/FAIL row. Failures prevent export. Earlier notebook assertions also stop execution on errors; use **Restart Kernel and Run All** for a complete run.

**Scope of PASS:** Faithful representation of the supplied DOCX's stored text and supported structures. It is not verification of legal correctness, website completeness, Word page rendering or a final legal hierarchy. Repeated source text is preserved; the test rejects accidental duplicate **records/locations**, not legitimate repeated paragraphs. Title/metadata values intentionally repeated in descriptors are checked as evidence-backed copies, not counted as additional body content.

### 8a. Establish the source and validation helpers

The source archive is opened read-only. We hash it before and after persistence. Validation compares original text without trimming or normalization. Small targeted failure cases exercise malformed input, changed text and a missing metadata structure; they do not change the real source.

In [ ]:
from tempfile import NamedTemporaryFile
import os

OUTPUT_PATH = ROOT / "data/processed/01_parsed_docx_structure.json"

def file_sha256(path):
    digest = sha256()
    with path.open("rb") as stream:
        for piece in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(piece)
    return digest.hexdigest()

source_archive_sha_before = file_sha256(ARCHIVE)
validation_docx_bytes = load_source(ARCHIVE, MEMBER)

def read_validation_body(data):
    with ZipFile(BytesIO(data)) as package:
        root = ET.fromstring(package.read("word/document.xml"))
    result = root.find("w:body", NS)
    if result is None:
        raise ValueError("Missing document body")
    return result

validation_body = read_validation_body(validation_docx_bytes)
validation_body_elements = list(validation_body)
validation_block_by_id = {b["id"]: b for b in corpus["blocks"]}
validation_paragraphs = all_paragraph_records(corpus["blocks"])
validation_record_by_id = {r["id"]: r for r in corpus["blocks"] + [
    p for b in corpus["blocks"] if b["type"] == "table" for p in table_paragraphs(b)]}
validation_rows = []

def require(condition, message):
    if not condition:
        raise AssertionError(message)

def run_validation(name, check):
    try:
        detail = check()
        result = {"check": name, "result": "PASS", "detail": detail}
    except Exception as exc:
        result = {"check": name, "result": "FAIL", "detail": f"{type(exc).__name__}: {exc}"}
    validation_rows.append(result)
    return result


def verify_paragraph_at(record, xml, part, path, position):
    require(record["type"] == "paragraph" and record["position"] == position, f"Paragraph position/type: {path}")
    require(record["source"] == {"part": part, "path": path}, f"Paragraph locator: {path}")
    require(record["raw_text"] == xml_visible_text(xml), f"Paragraph text changed: {path}")
    require([e["text"] for r in record["runs"] for e in r["events"] if e["type"] == "text"] ==
            [e.text or "" for e in xml.iter(W + "t")], f"Text fragments changed/dropped/duplicated: {path}")
    require(record["raw_text"] == "".join(r["raw_text"] for r in record["runs"]), f"Run text mismatch: {path}")
    explicit = xml.find("w:pPr/w:pStyle", NS)
    require(record["style"]["explicit_id"] == (explicit.get(W + "val") if explicit is not None else None), f"Style: {path}")
    require([(m["xml_tag"], m["kind"], m["clear"]) for m in record["break_markers"]] == [
        (e.tag.rsplit("}", 1)[-1], e.get(W + "type", "textWrapping"), e.get(W + "clear"))
        for e in xml.iter() if e.tag in {W + "br", W + "cr"}], f"Break events: {path}")


### 8b. Validate identity, order, text and coverage

Every top-level paragraph/table is compared at its original body position. Table rows/cells and their paragraph positions are checked without flattening tables into additional body text. This accounts for 31,055 body paragraphs plus 252 cell paragraphs, 21 tables, and one separate section-property record.

Boundary checks use the title/subtitle/metadata-table pattern and exact region delimiters in the source XML. The corpus title must remain front matter; 21 full-text markers and 19 optional reference-rule markers must be accounted for. All body blocks and nested cell paragraphs must have the expected membership/region.

In [ ]:
def check_source_identity():
    require(validation_docx_bytes == docx_bytes, "Source member changed since Step 1")
    require(corpus["source"] == {"archive": ARCHIVE.relative_to(ROOT).as_posix(), "member": MEMBER,
            "docx_bytes": len(validation_docx_bytes), "sha256": sha256(validation_docx_bytes).hexdigest()}, "Source provenance mismatch")
    require(corpus["parser"]["version"] == version("python-docx"), "Parser version mismatch")
    return "Exact archive member, byte size, SHA-256 and parser provenance agree"


def check_order_and_text():
    xml_items = [(i, e) for i, e in enumerate(validation_body) if e.tag in {W + "p", W + "tbl"}]
    require(len(xml_items) == len(corpus["blocks"]), "Body block count mismatch")
    for (position, xml), block in zip(xml_items, corpus["blocks"], strict=True):
        path = f"/body/children/{position}"
        if xml.tag == W + "p":
            verify_paragraph_at(block, xml, "word/document.xml", path, position)
        else:
            require(block["type"] == "table" and block["position"] == position, f"Table order at {path}")
            require(block["source"] == {"part": "word/document.xml", "path": path}, f"Table locator at {path}")
            for row_index, (row, xml_row) in enumerate(zip(block["rows"], xml.findall("w:tr", NS), strict=True)):
                require(row["row_index"] == row_index, f"Row order at {path}")
                for column_index, (cell, xml_cell) in enumerate(zip(row["cells"], xml_row.findall("w:tc", NS), strict=True)):
                    require(cell["column_index"] == column_index, f"Cell order at {path}")
                    cell_items = [(i, e) for i, e in enumerate(xml_cell) if e.tag == W + "p"]
                    for paragraph, (child_index, xml_p) in zip(cell["paragraphs"], cell_items, strict=True):
                        verify_paragraph_at(paragraph, xml_p, "word/document.xml",
                            f"{path}/rows/{row_index}/cells/{column_index}/children/{child_index}", child_index)
    observed = [sum(b["type"] == "paragraph" for b in corpus["blocks"]),
                sum(b["type"] == "table" for b in corpus["blocks"]),
                sum(len(b["rows"]) for b in corpus["blocks"] if b["type"] == "table"),
                sum(len(r["cells"]) for b in corpus["blocks"] if b["type"] == "table" for r in b["rows"]),
                len(validation_paragraphs),
                sum(m["kind"] == "page" for p in validation_paragraphs for m in p["break_markers"])]
    require(observed == [31055, 21, 126, 252, 31307, 41], f"Inspected source inventory changed: {observed}")
    require(len(corpus["blocks"]) + len(corpus["layout_markers"]) == len(validation_body), "Unaccounted body element")
    return "31,076 ordered body blocks; 31,307 body/cell paragraphs; 126 rows / 252 cells; 41 page breaks; exact text and locations"


def check_boundaries_and_regions():
    source_starts = []
    for i, xml in enumerate(validation_body_elements[:-2]):
        if style_id(xml) != "Heading1":
            continue
        subtitle, table = validation_body_elements[i + 1:i + 3]
        if subtitle.tag != W + "p" or table.tag != W + "tbl":
            continue
        labels = [text_preview(row.findall("w:tc", NS)[0]) for row in table.findall("w:tr", NS)]
        if labels == list(METADATA_LABELS):
            source_starts.append(i)
    documents = corpus["documents"]
    require(len(source_starts) == len(documents) == 21, "Expected 21 source documents")
    body_index_by_position = {b["position"]: i for i, b in enumerate(corpus["blocks"])}
    require(len({d["document_id"] for d in documents}) == 21, "Duplicate document IDs")
    for b in corpus["blocks"][:body_index_by_position[source_starts[0]]]:
        require(b["document_id"] is None and b["region"] == "front_matter", "Incorrect corpus front matter")
    full_count = reference_count = 0
    for ordinal, (document, source_start) in enumerate(zip(documents, source_starts, strict=True)):
        next_position = source_starts[ordinal + 1] if ordinal + 1 < len(source_starts) else len(validation_body_elements)
        start_index = body_index_by_position[source_start]
        end_index = body_index_by_position[next_position] if ordinal + 1 < len(source_starts) else len(corpus["blocks"])
        require(document["block_range"] == {"start_index": start_index, "end_index_exclusive": end_index,
                "first_body_position": source_start, "last_body_position": corpus["blocks"][end_index - 1]["position"]}, "Document range mismatch")
        require(document["source_titles"]["title"]["raw_text"] == text_preview(validation_body_elements[source_start]), "Document title mismatch")
        source_full = [i for i in range(source_start, next_position)
            if style_id(validation_body_elements[i]) == "Heading2" and text_preview(validation_body_elements[i]) == FULL_TEXT_MARKER]
        source_refs = [i for i in range(source_start, next_position)
            if style_id(validation_body_elements[i]) == "Heading2" and text_preview(validation_body_elements[i]) == REFERENCE_MARKER]
        require(len(source_full) == 1 and len(source_refs) <= 1, "Ambiguous source region delimiters")
        require(not source_refs or source_refs[0] < source_full[0], "Reversed reference/full-text markers")
        full_count += len(source_full); reference_count += len(source_refs)
        expected_evidence = {
            "title_block_id": corpus["blocks"][start_index]["id"],
            "subtitle_block_id": corpus["blocks"][start_index + 1]["id"],
            "metadata_table_block_id": corpus["blocks"][start_index + 2]["id"],
            "full_text_marker_block_id": corpus["blocks"][body_index_by_position[source_full[0]]]["id"],
            "reference_marker_block_id": corpus["blocks"][body_index_by_position[source_refs[0]]]["id"] if source_refs else None,
        }
        require(document["boundary_evidence"] == expected_evidence, "Boundary evidence mismatch")
        for b in corpus["blocks"][start_index:end_index]:
            region = ("full_text" if b["position"] >= source_full[0] else
                      "reference_rules" if source_refs and b["position"] >= source_refs[0] else "document_metadata")
            require(b["document_id"] == document["document_id"] and b["region"] == region, "Document/region coverage mismatch")
            if b["type"] == "table":
                require(all(p["document_id"] == b["document_id"] and p["region"] == region for p in table_paragraphs(b)), "Cell membership mismatch")
    require((full_count, reference_count) == (21, 19), "Region delimiter totals changed")
    require(dict(Counter(b["region"] for b in corpus["blocks"])) == corpus["diagnostics"]["region_counts"], "Region summary mismatch")
    return "21 contiguous source ranges, 24 front-matter blocks, 21 full-text / 19 reference markers; no gaps or overlaps"


### 8c. Validate metadata, provenance and structural evidence

IDs must be unique at canonical content locations; references may legitimately point to the same ID. Every document's six metadata rows and Step 7's categorized values must agree with their actual table paragraphs. Package properties remain separately scoped.

Legal-marker checks verify labels/spans, full-text membership and representative Article, Chapter, Section, `§`, and Annex evidence. They do not certify that every candidate is a legal provision. Footer fields and section-property records are checked separately from body text.

In [ ]:
def check_ids_and_auxiliary():
    canonical = list(validation_record_by_id.values()) + corpus["layout_markers"]
    for auxiliary in corpus["auxiliary_content"]:
        canonical += auxiliary["blocks"] + [p for b in auxiliary["blocks"] if b["type"] == "table" for p in table_paragraphs(b)] + auxiliary["layout_markers"]
        source_xml = auxiliary_xml[auxiliary["part"]]
        for record, xml in zip(all_paragraph_records(auxiliary["blocks"]), source_xml.iter(W + "p"), strict=True):
            require(record["raw_text"] == xml_visible_text(xml), "Auxiliary text mismatch")
            events = [e for r in record["runs"] for e in r["events"]]
            require([e["text"] for e in events if e["type"] == "field_instruction"] == [e.text or "" for e in xml.iter(W + "instrText")], "Footer field instruction lost")
            require([e["kind"] for e in events if e["type"] == "field_char"] == [e.get(W + "fldCharType") for e in xml.iter(W + "fldChar")], "Footer field order changed")
    expected_body_records = len(corpus["blocks"]) + sum(len(table_paragraphs(b)) for b in corpus["blocks"] if b["type"] == "table")
    require(len(validation_record_by_id) == expected_body_records, "Duplicate body/cell IDs")
    require(len({r["id"] for r in canonical}) == len(canonical), "Duplicate canonical IDs")
    require(len({(r["source"]["part"], r["source"]["path"]) for r in canonical}) == len(canonical), "Duplicate canonical source locations")
    require(all(r["id"] == base_record(r["type"], r["position"], r["source"]["part"], r["source"]["path"], corpus["source"]["sha256"])["id"] for r in canonical), "Unstable content/location IDs")
    source_sections = [(i, e) for i, e in enumerate(validation_body) if e.tag == W + "sectPr"]
    for marker, (position, xml) in zip(corpus["layout_markers"], source_sections, strict=True):
        require(marker["position"] == position and marker["xml"] == property_xml(xml), "Section properties changed")
    require({a["part"] for a in corpus["auxiliary_content"]} == set(auxiliary_xml), "Missing auxiliary part")
    return "Unique stable IDs/locations; footer PAGE events and section properties preserved separately"


def check_metadata_provenance():
    catalog = corpus["metadata_provenance"]
    require(catalog["generated"]["source_locator_and_fingerprint"] == corpus["source"], "Catalog file provenance mismatch")
    require(catalog["generated"]["parser"] == corpus["parser"], "Catalog parser provenance mismatch")
    row_count = 0
    for document, classified in zip(corpus["documents"], catalog["documents"], strict=True):
        require(document["document_id"] == classified["document_id"], "Classified document ID mismatch")
        require(document["source_titles"] == classified["source_extracted"]["titles"], "Classified titles mismatch")
        for title in document["source_titles"].values():
            require(title == text_with_evidence(validation_record_by_id[title["block_id"]]), "Title evidence mismatch")
        metadata = document["source_metadata"]
        table = validation_record_by_id[metadata["table_block_id"]]
        require(metadata["table_source"] == table["source"], "Metadata table location mismatch")
        require(len(metadata["rows"]) == len(table["rows"]) == 6, "Missing source metadata row")
        for entry, actual in zip(metadata["rows"], table["rows"], strict=True):
            row_count += 1
            require(entry["row_index"] == actual["row_index"], "Metadata row index mismatch")
            for field, index in [("label", 0), ("value", 1)]:
                require(entry[field + "_column_index"] == actual["cells"][index]["column_index"], "Metadata column index mismatch")
                require(entry[field + "_paragraphs"] == [text_with_evidence(p) for p in actual["cells"][index]["paragraphs"]], "Source metadata value/evidence modified")
        require(classified["source_extracted"]["metadata_fields"] == classified_source_rows(document), "Categorized source metadata modified")
        require(classified["structurally_interpreted"]["boundary_evidence"] == document["boundary_evidence"], "Interpreted boundary evidence mismatch")
    with ZipFile(BytesIO(validation_docx_bytes)) as package:
        core = ET.fromstring(package.read("docProps/core.xml"))
    for index, (record, xml) in enumerate(zip(catalog["source_extracted"]["package_properties"], core, strict=True)):
        require(record["qualified_name"] == xml.tag and record["raw_value"] == xml.text and record["attributes"] == dict(xml.attrib), "Package properties changed")
        require(record["source"] == {"part": "docProps/core.xml", "path": f"/coreProperties/children/{index}"}, "Package evidence mismatch")
    require(all(item["status"] == "deferred" and set(item) == {"field", "status", "reason"} for item in catalog["deferred"]), "Deferred metadata was inferred")
    require(any(item["field"] == "document_to_chunk_metadata_inheritance" for item in catalog["deferred"]), "Chunk inheritance decision missing from deferred list")
    require(row_count == 126, "Expected 126 original metadata rows")
    return "All 126 source metadata rows, titles, raw dates/notes, package properties and evidence agree; deferred decisions remain deferred"


def check_marker_evidence():
    markers = [m for b in corpus["blocks"] for m in b["structural_annotations"]["legal_marker_candidates"]]
    kinds = Counter(m["kind"] for m in markers)
    require(set(kinds) >= {"part", "chapter", "section", "article", "paragraph_sign", "annex"}, "Missing representative legal-marker kinds")
    for m in markers:
        block = validation_record_by_id[m["block_id"]]
        lo, hi = m["character_span"]
        require(block["raw_text"][lo:hi] == m["matched_label"] and m["source"] == block["source"], "Marker source/span mismatch")
        require(m["status"] == "candidate" and block["region"] == "full_text" and m["document_id"] == block["document_id"], "Marker status/membership mismatch")
        require(detect_legal_candidate(block, {"region": block["region"], "document_id": block["document_id"]}) == m, "Candidate rule evidence mismatch")
    require(dict(kinds) == corpus["diagnostics"]["candidate_marker_counts"], "Marker count summary mismatch")
    return f"{len(markers)} candidate labels retain source spans/rules; Article, Chapter, Section, § and Annex examples available"


def check_targeted_failures():
    def expect_failure(action, errors):
        try:
            action()
        except errors:
            return
        raise AssertionError("Malformed fixture was silently accepted")
    expect_failure(lambda: read_validation_body(b"not a DOCX"), BadZipFile)
    buffer = BytesIO()
    with ZipFile(buffer, "w") as z:
        z.writestr("word/document.xml", "<broken")
    expect_failure(lambda: read_validation_body(buffer.getvalue()), ET.ParseError)
    altered = deepcopy(corpus["blocks"][0]); altered["raw_text"] += " "
    expect_failure(lambda: verify_paragraph_at(altered, validation_body_elements[0], "word/document.xml", "/body/children/0", 0), AssertionError)
    source = corpus["documents"][0]
    first = source["block_range"]["start_index"]
    minimal = deepcopy(corpus["blocks"][first:first + 3])
    minimal[2]["rows"][0]["cells"][0]["paragraphs"][0]["raw_text"] = "unexpected metadata label"
    expect_failure(lambda: discover_document_starts(minimal), ValueError)
    # Steps 4–7 already exercise mixed order, inline breaks, empty/cell paragraphs,
    # missing/duplicate region markers and unknown/duplicate metadata rows.
    return "Four targeted failures caught: invalid DOCX, malformed XML, changed text, missing metadata pattern; earlier fixture checks also passed"

for name, check in [
    ("Source identity/provenance", check_source_identity),
    ("Paragraph/table order and exact source text", check_order_and_text),
    ("21 document boundaries and complete region coverage", check_boundaries_and_regions),
    ("Stable IDs, auxiliary fields and layout", check_ids_and_auxiliary),
    ("Source metadata and evidence fidelity", check_metadata_provenance),
    ("Representative structural-marker evidence", check_marker_evidence),
    ("Malformed/missing/changed input guards", check_targeted_failures),
]:
    run_validation(name, check)
show_rows(validation_rows)


### 8d. Report ambiguity and show source previews

Unclassified headings, repeated heading labels and provisional legal markers are **review items**, not automatically parsing failures. Their text and locations remain available. No classification is invented to make the report look complete.

Inspect a representative marker from each requested category, a suffix-bearing article, a long paragraph, and the end of the final source. The last preview deliberately retains website/footer-like text. A source note claiming website boilerplate was removed does not justify removing text ourselves.

Automated fidelity checks cannot replace your review of semantic boundaries or legal meaning. These outputs are previews for that review; PASS below refers only to the stated automated checks.

In [ ]:
stage1_marker_examples = []
for kind in ["article", "chapter", "section", "paragraph_sign", "annex"]:
    example = next((m for b in corpus["blocks"] for m in b["structural_annotations"]["legal_marker_candidates"] if m["kind"] == kind), None)
    if example:
        block = validation_block_by_id[example["block_id"]]
        stage1_marker_examples.append({"kind": kind, "body position": block["position"],
                                      "source label": example["matched_label"], "source text preview": block["raw_text"][:350]})
show_rows(stage1_marker_examples)
review_headings = [b for b in corpus["blocks"] if b["type"] == "paragraph" and b["region"] == "full_text"
                   and heading_level(b) in {2, 3} and b["raw_text"] != FULL_TEXT_MARKER
                   and not b["structural_annotations"]["legal_marker_candidates"]]
heading_groups = defaultdict(list)
for b in review_headings:
    heading_groups[b["raw_text"]].append(b["position"])
show_rows([{"unclassified heading": b["raw_text"], "body position": b["position"], "style": b["style"]["resolved_id"]}
           for b in review_headings[:8]])
show_rows([{"repeated source heading": text, "occurrences": len(positions), "first positions": positions[:6]}
           for text, positions in heading_groups.items() if len(positions) > 1][:5])
long_paragraph = max(validation_paragraphs, key=lambda p: len(p["raw_text"]))
suffix_paragraph = next(p for p in validation_paragraphs if p["raw_text"].startswith("Чл. 2а."))
last_document = corpus["documents"][-1]
last_span = corpus["blocks"][last_document["block_range"]["start_index"]:last_document["block_range"]["end_index_exclusive"]]
last_nonempty = [b for b in last_span if b["type"] == "paragraph" and b["raw_text"]][-3:]
show_rows([{"sample": label, "body position": p["position"], "characters": len(p["raw_text"]), "preview": p["raw_text"][:500]}
           for label, p in [("long paragraph", long_paragraph), ("article suffix/Cyrillic", suffix_paragraph)] + [("final source ending", p) for p in last_nonempty]])
show_full_record({"source": long_paragraph["source"], "raw_text": long_paragraph["raw_text"]}, "Complete longest paragraph, unchanged")

stage1_review_items = [
    {"code": "unclassified_headings", "count": len(review_headings),
     "message": "Preserved without guessing a legal role.",
     "block_ids": [b["id"] for b in review_headings]},
    {"code": "repeated_heading_labels", "count": sum(len(v) > 1 for v in heading_groups.values()),
     "message": "Repeated labels at distinct source locations are retained; no deduplication or final hierarchy."},
    {"code": "candidate_markers_only", "message": "Candidate legal labels do not establish provision semantics or legal hierarchy."},
    {"code": "source_page_navigation_retained", "message": "Source-page navigation/footer-like text remains unchanged; cleaning is deferred."},
    {"code": "rendered_layout_not_validated", "message": "No physical page numbers or Word visual-layout reconstruction."},
    {"code": "legal_accuracy_not_validated", "message": "Fidelity is to this DOCX, not current law or completeness of original websites."},
    {"code": "chunk_inheritance_undecided", "message": "Chunk metadata and document-to-chunk inheritance remain deferred."},
]
show_rows([{"review item": item["code"], "detail": item["message"]} for item in stage1_review_items])
validation_passed = len(validation_rows) == 7 and all(row["result"] == "PASS" for row in validation_rows)
corpus["diagnostics"]["stage1_validation"] = {
    "result": "PASS" if validation_passed else "FAIL", "scope": "Stored DOCX text/structure fidelity; not legal accuracy or rendered layout",
    "checks": deepcopy(validation_rows), "review_items": deepcopy(stage1_review_items),
}
if not validation_passed:
    raise RuntimeError("Stage 1 validation failed. No artifact will be written; inspect FAIL rows above.")
validated_corpus_digest = record_digest(corpus)
print("PASS: all required automated checks. Review items retained without inference or repair.")


### 8e. Persist and reload the validated representation

**What/how:** Write UTF-8 JSON with readable indentation to a temporary file in `data/processed/`, then atomically replace only `01_parsed_docx_structure.json`. This avoids a partially written final file. The original archive, DOCX bytes and existing PDF/TXT artifact are untouched.

A guard checks that the corpus has not changed since validation. Reload the final file and require exact equality with the in-memory `corpus`. Compare the source archive hash before/after writing. On rerun, compare the output hash with any previous artifact; identical bytes demonstrate repeatable output for unchanged inputs/code. A changed artifact is reported as changed, not silently labeled repeatable.

**Trade-off:** Readable JSON is larger than the compressed DOCX because it includes text, run structure, provenance and annotations. It remains a simple local intermediate artifact, not a database. No timestamps are added to the validation report, so unchanged runs can produce identical output.

This is the only cell in the notebook that writes the processed corpus. Rerunning it replaces that derived file after the validation guard passes.

In [ ]:
require(validation_passed and corpus["diagnostics"]["stage1_validation"]["result"] == "PASS", "Validation must pass before persistence")
require(record_digest(corpus) == validated_corpus_digest, "Corpus changed after validation; rerun Step 8 checks")
require(file_sha256(ARCHIVE) == source_archive_sha_before, "Source archive changed during this run")
require(not OUTPUT_PATH.is_symlink(), "Processed output must not be a symlink")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
previous_output_sha = file_sha256(OUTPUT_PATH) if OUTPUT_PATH.exists() else None
temporary_path = None
try:
    with NamedTemporaryFile("w", encoding="utf-8", dir=OUTPUT_PATH.parent,
                            prefix=".stage01-", suffix=".json.tmp", delete=False) as stream:
        temporary_path = Path(stream.name)
        json.dump(corpus, stream, ensure_ascii=False, indent=2, allow_nan=False)
        stream.write("\n")
    os.replace(temporary_path, OUTPUT_PATH)
finally:
    if temporary_path is not None and temporary_path.exists():
        temporary_path.unlink()

with OUTPUT_PATH.open("r", encoding="utf-8") as stream:
    reloaded_corpus = json.load(stream)
require(reloaded_corpus == corpus, "FAIL: saved JSON does not exactly equal the in-memory corpus")
del reloaded_corpus
require(file_sha256(ARCHIVE) == source_archive_sha_before, "FAIL: source archive changed during persistence")
output_sha = file_sha256(OUTPUT_PATH)
repeatability_result = ("NOT CHECKED — first saved artifact" if previous_output_sha is None else
                        "PASS — identical to previous artifact" if output_sha == previous_output_sha else
                        "CHANGED — compare source/code/parser versions before claiming repeatability")
persistence_rows = [
    {"check": "UTF-8 JSON save/reload equality", "result": "PASS"},
    {"check": "Original source archive unchanged", "result": "PASS"},
    {"check": "Repeat-run output comparison", "result": repeatability_result},
]
show_rows(persistence_rows)
print("Saved:", OUTPUT_PATH.relative_to(ROOT))
print("Bytes:", OUTPUT_PATH.stat().st_size, "SHA-256:", output_sha)


### Final Stage 1 summary

The summary below records the actual automated validation and persistence results. Known limitations and deferred decisions remain visible in the notebook and artifact diagnostics. The 31,076 structural blocks are top-level body records; the 252 table-cell paragraphs remain nested and are counted separately in fidelity checks.

In [ ]:
stage1_summary = {
    "source file": corpus["source"]["member"],
    "source archive": corpus["source"]["archive"],
    "documents": len(corpus["documents"]),
    "structural body blocks": len(corpus["blocks"]),
    "body paragraphs / tables": f"{sum(b['type'] == 'paragraph' for b in corpus['blocks'])} / {sum(b['type'] == 'table' for b in corpus['blocks'])}",
    "metadata preservation": "PASS — all 126 source metadata rows and evidence preserved",
    "output file": OUTPUT_PATH.relative_to(ROOT).as_posix(),
    "output bytes": OUTPUT_PATH.stat().st_size,
    "validation result": "PASS — automated source fidelity and exact JSON reload equality",
    "repeatability": repeatability_result,
    "known limitations": f"{len(review_headings)} unclassified headings retained; candidate legal roles only; website text retained; no rendered pages or legal accuracy verification",
    "deferred decisions": "Metadata normalization/inference, final legal hierarchy, chunk design and document-to-chunk metadata inheritance",
}
show_rows([{"Stage 1 item": key, "result": value} for key, value in stage1_summary.items()])


## Stop — Stage 1 complete; review required before the next stage

The validated Stage 1 representation is saved at `data/processed/01_parsed_docx_structure.json`. Review the PASS/FAIL table, preserved ambiguity report, source previews, and final summary before approving further work.

No cleaning, chunking, embeddings, vector storage, retrieval, RAG, or evaluation was performed. Original source data remains unchanged. No final legal hierarchy or chunk-metadata inheritance policy has been selected. **Do not begin the next stage without your review and approval.**